# Lab 3: Autoencoders

This lab covers **autoencoders**: an encoder compresses the input into a code, a decoder reconstructs the input from it, and both are trained together. All of them are encoder -> bottleneck -> decoder; they differ in how the bottleneck is created, what is decoded, and how they are trained. For each type: a short theory recap, a pretrained model when one exists, and an implementation from scratch for a practical task.

1. **Tabular autoencoders** - from PCA to deep count autoencoders on single-cell gene expression
2. **Autoencoders for embeddings** - compressing DINOv3 image embeddings
3. **Denoising autoencoders** - a pretrained image denoiser, and a convolutional denoiser for CT scans
4. **Masked autoencoders** - a pretrained MAE, and MAE pretraining of a small ViT
5. **Sparse autoencoders** - interpreting and steering an LLM with pretrained SAEs, and our own SAE on DINOv3 features
6. **U-Net** - retinal vessel segmentation, from an encoder-decoder without skip connections to a modern U-Net, and a denoising U-Net

In exercises, fill in the `...` gaps; the following cells check your solution.

**Hardware** - every model trains in at most a few minutes on a GPU; free Google Colab T4 is enough (*Runtime -> Change runtime type -> T4 GPU*). Apple GPUs (MPS) are used automatically. CPU works for Part 1, but the rest will be slow.

# Environment setup

Use a local environment with the [uv](https://docs.astral.sh/uv/) package manager (tested), or [Google Colab](https://colab.research.google.com/). Packages installed manually with `pip` may break because of incompatible versions.

### Local environment

1. Install uv [following the documentation](https://docs.astral.sh/uv/getting-started/installation/), if you don't have it yet
2. In the directory of this lab, install the dependencies from `uv.lock`: `uv sync`
3. Run Jupyter Notebook: `uv run jupyter notebook`

### Google Colab

Upload `requirements.txt` from the directory of this lab to `/content` and run `!pip install -r requirements.txt` in the cell below. If Colab asks to restart the session, do it and continue with the next cell.

### Models and datasets

Datasets are downloaded to `data` (about 1.5 GB), models from the [HuggingFace Hub](https://huggingface.co/) to `~/.cache/huggingface` (about 6 GB). No account is needed.

In [ ]:
# uncomment and run if using Google Colab
# !pip install -r requirements.txt

In [ ]:
import gc
import math
import os
import time
import warnings
from collections.abc import Callable

import anndata
import deepinv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import transformers
from matplotlib_inline.backend_inline import set_matplotlib_formats
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from torchvision.datasets.utils import download_url

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# 16-bit precision: bfloat16 on modern NVIDIA GPUs, float16 on older GPUs
# (e.g. Colab T4) and Apple GPUs, float32 on CPU
if device.type == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False):
    HALF_DTYPE = torch.bfloat16
elif device.type in ("cuda", "mps"):
    HALF_DTYPE = torch.float16
else:
    HALF_DTYPE = torch.float32
# mixed precision training on NVIDIA GPUs
USE_AMP = device.type == "cuda"

print(
    f"PyTorch {torch.__version__}, timm {timm.__version__}, "
    f"transformers {transformers.__version__}, device: {device}, 16-bit dtype: {HALF_DTYPE}"
)

In [ ]:
# datasets and computed embeddings are saved here
DATA_ROOT = "data"
os.makedirs(DATA_ROOT, exist_ok=True)


def download(url: str, filename: str) -> str:
    # downloads the file once, returns its path
    path = os.path.join(DATA_ROOT, filename)
    if not os.path.exists(path):
        download_url(url, DATA_ROOT, filename)
    return path


def free_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()


def autocast() -> torch.autocast:
    return torch.autocast(device_type=device.type, dtype=HALF_DTYPE, enabled=USE_AMP)


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


transformers.set_seed(0)
plt.rcParams.update(
    {"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False}
)
# JPEG instead of PNG figures to make notebook smaller
set_matplotlib_formats("jpeg", pil_kwargs={"quality": 90})

# hide progress bars and informational messages
transformers.logging.disable_progress_bar()
transformers.logging.set_verbosity_error()

# Part 1: Tabular autoencoders

An autoencoder has an **encoder** $f$, mapping the input $x \in \mathbb{R}^D$ to a code $z \in \mathbb{R}^k$, and a **decoder** $g$, reconstructing $\hat{x} = g(z)$, trained to minimize e.g. $\|x - \hat{x}\|^2$. With $k \ll D$, the code must keep only what is most informative. We start with tabular data (rows are samples, columns are features) and the simplest autoencoders: MLPs.

## 1.1 Single-cell gene expression

**Single-cell RNA sequencing** counts the captured mRNA molecules of every gene in individual cells, giving a table of counts, cells x genes. It's one of the main data types of modern biology, used e.g. to build cell atlases of organs and find cell types involved in diseases.

We use the healthy human bone marrow atlas of [Triana et al. (2021)](https://www.nature.com/articles/s41556-021-00681-2) from [CELLxGENE](https://cellxgene.cziscience.com/): 49k cells from 6 donors, 461 genes and 39 cell types annotated by experts. Bone marrow produces all blood cells: stem cells divide into progenitors, which mature into red blood cells, platelets and immune cells (T and B lymphocytes, NK cells, monocytes, ...). It's a typical scientific tabular dataset: noisy, sparse and nonlinear.

In [ ]:
SC_URL = "https://datasets.cellxgene.cziscience.com/9388f207-6d35-47a7-bbb5-7fdf7b81dc5b.h5ad"
adata = anndata.read_h5ad(download(SC_URL, "bone_marrow.h5ad"))

# raw counts: a sparse matrix (cells, genes)
counts = adata.raw.X.toarray().astype(np.float32)
genes = adata.var["feature_name"].astype(str).values
cell_types = adata.obs["cell_type"].astype(str).values
cell_type_names, y_cells = np.unique(cell_types, return_inverse=True)

print(
    f"{counts.shape[0]} cells x {counts.shape[1]} genes, {len(cell_type_names)} cell types"
)
print(
    f"zeros: {100 * (counts == 0).mean():.0f}%, median counts per cell: {np.median(counts.sum(axis=1)):.0f}"
)
pd.DataFrame(counts[:5, :8].astype(int), columns=genes[:8], index=cell_types[:5])

Most counts are zero, as only some mRNA molecules are captured, and cells differ in the total number captured (the **library size**), a technical effect. Standard preprocessing: scale every cell to 10,000 counts, take $\log(1 + x)$, and standardize every gene.

**Plasma cells** (antibody-producing B cells) are held out from training as a novel cell type for anomaly detection. The other cells are split randomly 80/20 into train and test.

In [ ]:
library_size = counts.sum(axis=1, keepdims=True)
log_expression = np.log1p(counts / library_size * 1e4)
X_cells = (log_expression - log_expression.mean(axis=0)) / log_expression.std(axis=0)
NUM_GENES = X_cells.shape[1]

rng = np.random.default_rng(0)
is_novel = cell_types == "plasma cell"
permutation = rng.permutation(len(X_cells))
test_cells = permutation[: len(X_cells) // 5]
train_cells = permutation[len(X_cells) // 5 :]
train_cells = train_cells[~is_novel[train_cells]]  # no plasma cells in training
test_known = test_cells[~is_novel[test_cells]]
print(
    f"train: {len(train_cells)} cells, test: {len(test_known)} known + "
    f"{is_novel[test_cells].sum()} novel (plasma) cells"
)

# everything on the GPU, it's small: standardized log expression, counts, library sizes
X_cells_t = torch.tensor(X_cells, dtype=torch.float32, device=device)
counts_t = torch.tensor(counts, device=device)

### Evaluation and training tools

We evaluate the codes, as in **unsupervised representation learning**:

1. **kNN accuracy** - do the 15 nearest neighbors of a test cell in the code space have its cell type? This checks whether similar cells get similar codes.
2. **few-label accuracy** - logistic regression on codes of only 10 labeled cells per type. Labeling cells needs experts, so this is a realistic scenario.

`fit_autoencoder` trains a model with a given loss on batches of training cells. `evaluate_codes` saves the results of a step and plots all steps so far.

This section is collapsed: you only need to run it.

In [ ]:
CODE_SIZE = 16
SC_EPOCHS = 40
cell_results = {}

# 10 labeled cells of every type
few_labels = np.concatenate(
    [
        rng.permutation(train_cells[y_cells[train_cells] == c])[:10]
        for c in np.unique(y_cells[train_cells])
    ]
)


def fit_autoencoder(
    model: nn.Module, loss_fn: Callable, epochs: int = SC_EPOCHS, lr: float = 1e-3
) -> nn.Module:
    # loss_fn(model, x, counts) for a batch of standardized expression x and raw counts
    transformers.set_seed(0)
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    train_cells_t = torch.tensor(train_cells, device=device)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        opt, lr, total_steps=epochs * math.ceil(len(train_cells) / 256)
    )
    start = time.time()
    for epoch in range(epochs):
        model.train()
        order = train_cells_t[torch.randperm(len(train_cells_t), device=device)]
        for batch in order.split(256):
            loss = loss_fn(model, X_cells_t[batch], counts_t[batch])
            opt.zero_grad()
            loss.backward()
            opt.step()
            scheduler.step()
    print(f"trained in {time.time() - start:.0f} s, last batch loss {loss.item():.3f}")
    return model.eval()


@torch.no_grad()
def encode_cells(model: nn.Module) -> np.ndarray:
    return model.encoder(X_cells_t).cpu().numpy()


@torch.no_grad()
def test_mse(model: nn.Module) -> float:
    x = X_cells_t[test_known]
    return F.mse_loss(model(x), x).item()


def pca_test_mse(pca: PCA) -> float:
    x = X_cells[test_known]
    return np.mean((pca.inverse_transform(pca.transform(x)) - x) ** 2)


def cell_knn_accuracy(codes: np.ndarray) -> float:
    knn = KNeighborsClassifier(15).fit(codes[train_cells], y_cells[train_cells])
    return knn.score(codes[test_known], y_cells[test_known])


def evaluate_codes(name: str, codes: np.ndarray) -> None:
    probe = LogisticRegression(max_iter=3000).fit(
        codes[few_labels], y_cells[few_labels]
    )
    cell_results[name] = {
        "kNN accuracy": cell_knn_accuracy(codes),
        "few-label accuracy": probe.score(codes[test_known], y_cells[test_known]),
    }
    print(", ".join(f"{m}: {100 * v:.1f}%" for m, v in cell_results[name].items()))
    plot_results(cell_results, f"cell types from {CODE_SIZE}-dimensional codes")


def plot_results(results: dict, title: str, xlim: tuple | None = None) -> None:
    # horizontal bars: one panel per metric, one bar per step
    names = list(results)
    metrics = list(results[names[0]])
    fig, axes = plt.subplots(
        1, len(metrics), figsize=(5 * len(metrics), 0.4 * len(names) + 1), sharey=True
    )
    for ax, metric in zip(np.atleast_1d(axes), metrics):
        values = [100 * results[n][metric] for n in names]
        bars = ax.barh(names, values, color="#7d6cc4")
        ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=9)
        ax.set_xlim(xlim or (max(0, min(values) - 15), 100))
        ax.set_title(metric, fontsize=10)
    np.atleast_1d(axes)[0].invert_yaxis()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## 1.2 PCA is a linear autoencoder

PCA projects centered data onto the $k$ directions of the largest variance, the columns of an orthonormal $W \in \mathbb{R}^{D \times k}$: encoding is $z = W^\top x$, decoding $\hat{x} = W z$, and it minimizes $\|X - XWW^\top\|^2$. So PCA is a **linear autoencoder** with tied weights. Two linear layers trained with MSE learn the same subspace ([Plaut, 2018](https://arxiv.org/abs/1804.10253)), but not necessarily the same basis: their codes are an invertible linear transformation of the PCA codes. Our baseline is PCA with 16 components.

In [ ]:
pca = PCA(CODE_SIZE).fit(X_cells[train_cells])
pca_mse = pca_test_mse(pca)
print(f"PCA: test reconstruction MSE {pca_mse:.3f}")
evaluate_codes("1. PCA", pca.transform(X_cells))

### Exercise 1: Linear autoencoder

Implement `LinearAutoencoder`: `self.encoder` and `self.decoder` are single linear layers (`num_features -> code_size -> num_features`), and `forward` returns the reconstruction. Then implement `mse_loss(model, x, counts)`, the MSE between `model(x)` and `x`; the optional `counts` is unused and only matches the other losses.

In [ ]:
class LinearAutoencoder(nn.Module):
    def __init__(self, num_features: int, code_size: int) -> None:
        super().__init__()
        ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        ...


def mse_loss(
    model: nn.Module, x: torch.Tensor, counts: torch.Tensor | None = None
) -> torch.Tensor:
    ...

In [ ]:
from scipy.linalg import subspace_angles

model = LinearAutoencoder(NUM_GENES, CODE_SIZE)
assert model(torch.randn(3, NUM_GENES)).shape == (3, NUM_GENES)
assert count_params(model) == 2 * NUM_GENES * CODE_SIZE + CODE_SIZE + NUM_GENES

linear_ae = fit_autoencoder(model, mse_loss)
linear_mse = test_mse(linear_ae)
print(
    f"test reconstruction MSE: linear autoencoder {linear_mse:.3f}, PCA {pca_mse:.3f}"
)
assert linear_mse < 1.03 * pca_mse, "the linear autoencoder should be as good as PCA"

# angles between the subspaces spanned by decoder columns and principal components
decoder_columns = linear_ae.decoder.weight.detach().cpu().numpy()
angles = np.degrees(subspace_angles(decoder_columns, pca.components_.T))
print(f"angles between the subspaces (degrees): {np.round(angles, 1)}")
print(
    f"variance of the last PCA components: {np.round(pca.explained_variance_[-4:], 2)}"
)
assert np.median(angles) < 5, "the linear autoencoder should learn the PCA subspace"

evaluate_codes("2. linear autoencoder", encode_cells(linear_ae))
print("OK")

The same error and almost the same subspace: most directions agree within a degree or two. The rest are PCA components with almost equal variance, whose rotation barely changes the loss, so gradient descent converges very slowly along them. The codes are equally good, but linear models can only "flatten" the data, while real data manifolds are curved.

## 1.3 Nonlinear autoencoder

Deep autoencoders, with nonlinear MLP encoders and decoders, compress much better than PCA ([Hinton and Salakhutdinov, 2006](https://www.science.org/doi/10.1126/science.1127647)). We use the modern MLP block: linear layer, LayerNorm and GELU.

### Exercise 2: MLP autoencoder

1. `mlp(dims)`: an `nn.Sequential` of linear layers `dims[0] -> dims[1] -> ... -> dims[-1]`, with `nn.LayerNorm` and `nn.GELU` after every linear layer except the last one
2. `MLPAutoencoder`: the encoder is an MLP `num_features -> hidden -> code_size`, and the decoder mirrors it, `code_size -> reversed(hidden) -> num_features`; `hidden` is a tuple of layer sizes, e.g. `(512, 256)`

In [ ]:
def mlp(dims: list[int]) -> nn.Sequential:
    ...


class MLPAutoencoder(nn.Module):
    def __init__(
        self, num_features: int, code_size: int, hidden: tuple[int, ...] = (512, 256)
    ) -> None:
        super().__init__()
        ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))

In [ ]:
layers = mlp([10, 8, 4])
assert [type(l) for l in layers] == [nn.Linear, nn.LayerNorm, nn.GELU, nn.Linear], (
    f"got {[type(l).__name__ for l in layers]}"
)
model = MLPAutoencoder(NUM_GENES, CODE_SIZE)
assert model.encoder[0].in_features == NUM_GENES
assert model.encoder[-1].out_features == model.decoder[0].in_features == CODE_SIZE
assert model.decoder[-1].out_features == NUM_GENES
assert count_params(model) == 747485, f"expected 747485, got {count_params(model)}"

mlp_ae = fit_autoencoder(model, mse_loss)
mlp_mse = test_mse(mlp_ae)
print(f"test reconstruction MSE: MLP autoencoder {mlp_mse:.3f}, PCA {pca_mse:.3f}")
evaluate_codes("3. MLP autoencoder", encode_cells(mlp_ae))
print("OK")

The reconstruction error is about 30% lower than PCA's, but the codes are even a bit worse for cell types. MSE weighs every deviation equally, so the model also spends its capacity on reconstructing the sampling noise of the counts.

## 1.4 Count likelihood: deep count autoencoder

MSE on log values is a Gaussian likelihood, a poor model for counts. Gene counts follow the **negative binomial (NB)** distribution with mean $\mu$ and dispersion $\theta$ (variance $\mu + \mu^2/\theta$, more than Poisson). Single-cell autoencoders like [DCA](https://www.nature.com/articles/s41467-018-07931-2) and [scVI](https://www.nature.com/articles/s41592-018-0229-2) use it as the reconstruction loss:

* the decoder predicts each gene's fraction of the cell's counts (softmax over genes); times the library size, this gives $\mu$
* $\theta$ is a learned parameter per gene
* the loss is the negative log-likelihood $-\log \mathrm{NB}(\mathrm{counts} \mid \mu, \theta)$

The input is still the standardized log expression. As $\mu$ includes the library size, the code doesn't need to store it.

### Exercise 3: Negative binomial loss

1. In `CountAutoencoder`, add `self.log_theta`: an `nn.Parameter` with the log-dispersion per gene, initialized with zeros. `forward` (given) returns $\mu$
2. `nb_loss(model, x, counts)`: compute the library sizes (sum of counts of every cell, `keepdim=True`) and $\mu$, and return the mean negative log-likelihood of counts, using [`torch.distributions.NegativeBinomial`](https://docs.pytorch.org/docs/stable/distributions.html#negativebinomial) with `total_count` $= \theta$ and `logits` $= \log \mu - \log \theta$ (its mean is then exactly $\mu$)

In [ ]:
class CountAutoencoder(MLPAutoencoder):
    def __init__(self, num_features: int, code_size: int, **kwargs) -> None:
        super().__init__(num_features, code_size, **kwargs)
        ...

    def forward(self, x: torch.Tensor, library_size: torch.Tensor) -> torch.Tensor:
        # mean counts: fractions of the library size for every gene
        return self.decoder(self.encoder(x)).softmax(dim=-1) * library_size


def nb_loss(model: nn.Module, x: torch.Tensor, counts: torch.Tensor) -> torch.Tensor:
    ...

In [ ]:
model = CountAutoencoder(NUM_GENES, CODE_SIZE)
assert isinstance(model.log_theta, nn.Parameter)
assert model.log_theta.shape == (NUM_GENES,) and torch.all(model.log_theta == 0)

# compare with the formula: for theta = 1, log NB(c | mu) = log(1 / (1 + mu)) + c log(mu / (1 + mu))
x, c = X_cells_t[:4].cpu(), counts_t[:4].cpu()
mean = model(x, c.sum(dim=1, keepdim=True))
expected = -(torch.log(1 / (1 + mean)) + c * torch.log(mean / (1 + mean))).mean()
assert torch.allclose(nb_loss(model, x, c), expected, atol=1e-4), (
    "nb_loss differs from the NB negative log-likelihood"
)

count_ae = fit_autoencoder(model, nb_loss)
evaluate_codes("4. count autoencoder", encode_cells(count_ae))
print("OK")

A better model of the data gives much better codes: a few points more in kNN accuracy and about 10 with few labels, now clearly better than PCA.

## 1.5 Denoising the input

Even with a narrow bottleneck, an autoencoder can learn shortcuts, e.g. memorize individual genes. A simple regularizer is to corrupt the input and reconstruct the clean data: a **denoising autoencoder** (Part 3). Like DCA and scVI, we use **dropout on the input genes**, which mimics the main noise of single-cell data, uncaptured molecules. The loss still compares the output with the full counts.

### Exercise 4: Input dropout

Implement `DenoisingCountAutoencoder`: insert `nn.Dropout(dropout)` as the first layer of the encoder ([`nn.Sequential.insert`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html#torch.nn.Sequential.insert)). Dropout is active only in training mode, so after training the codes are computed from the complete input.

In [ ]:
class DenoisingCountAutoencoder(CountAutoencoder):
    def __init__(
        self, num_features: int, code_size: int, dropout: float = 0.2, **kwargs
    ) -> None:
        super().__init__(num_features, code_size, **kwargs)
        ...

In [ ]:
model = DenoisingCountAutoencoder(NUM_GENES, CODE_SIZE)
assert isinstance(model.encoder[0], nn.Dropout) and model.encoder[0].p == 0.2
assert isinstance(model.encoder[1], nn.Linear), "dropout goes before the first layer"
x = torch.randn(4, NUM_GENES)
assert not torch.allclose(model.train().encoder(x), model.encoder(x)), (
    "dropout should be active in training mode"
)
assert torch.allclose(model.eval().encoder(x), model.encoder(x)), "and off in eval mode"

denoising_ae = fit_autoencoder(model, nb_loss)
evaluate_codes("5. + input dropout", encode_cells(denoising_ae))
print("OK")

A small but consistent gain. Overall, the deep count autoencoder with input dropout beats PCA by about 4 points in kNN accuracy and 7 with few labels, trained in seconds without labels.

## 1.6 Applications: compression, visualization, novelty detection

#### Compression

How many dimensions do we need? We compare the reconstruction error of PCA and the MLP autoencoder (MSE loss) for different code sizes.

In [ ]:
code_sizes = [2, 4, 8, 16, 32]
compression_mse = {"PCA": [], "MLP autoencoder": []}
for k in code_sizes:
    compression_mse["PCA"].append(pca_test_mse(PCA(k).fit(X_cells[train_cells])))
    model = fit_autoencoder(MLPAutoencoder(NUM_GENES, k), mse_loss)
    compression_mse["MLP autoencoder"].append(test_mse(model))

fig, ax = plt.subplots(figsize=(6, 3.5))
for name, values in compression_mse.items():
    ax.plot(code_sizes, values, "o-", lw=2, label=name)
ax.set_xscale("log", base=2)
ax.set_xticks(code_sizes, code_sizes)
ax.set(
    xlabel="code size", ylabel="test reconstruction MSE",
    title=f"Compressing {NUM_GENES} genes",
)  # fmt: skip
ax.legend()
plt.tight_layout()
plt.show()

The autoencoder with 2 numbers per cell matches PCA with 16, and the gap stays large for all sizes: the data lies near a curved low-dimensional manifold (the **manifold hypothesis** from the lecture).

#### Visualization

With a 2-dimensional code, an autoencoder is a visualization method. Unlike t-SNE or UMAP, it's a parametric mapping: the encoder embeds new cells instantly. Below, PCA and the count autoencoder with 2-dimensional codes, colored by the main lineage.

In [ ]:
def plot_codes(codes: dict, groups: np.ndarray, accuracies: dict, q: float) -> None:
    # one panel per method with 2-dimensional codes, colored by the group of every point
    fig, axes = plt.subplots(1, len(codes), figsize=(14, 6))
    for ax, (name, xy) in zip(axes, codes.items()):
        for i, group in enumerate(np.unique(groups)):
            ax.scatter(*xy[groups == group].T, s=3, color=f"C{i}", label=group)
        # zoom in on the bulk of the points
        low, high = np.quantile(xy, [q, 1 - q], axis=0)
        ax.set(xlim=(low[0], high[0]), ylim=(low[1], high[1]), xticks=[], yticks=[])
        ax.set_title(
            f"{name}, 2-dimensional codes\nkNN accuracy {100 * accuracies[name]:.0f}%"
        )
    axes[-1].legend(markerscale=5, loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()


LINEAGES = {  # keyword in the cell type name -> lineage, first match wins
    "erythro": "erythroid + megakaryocytes",
    "megakaryocyte": "erythroid + megakaryocytes",
    "plasma": "plasma cells",
    "B cell": "B cells",
    "B-II": "B cells",
    "T cell": "T cells",
    "natural killer": "NK cells",
    "monocyte": "monocytes",
    "dendritic": "dendritic cells",
    "myelocyte": "granulocytic",
    "basophil": "granulocytic",
    "progenitor": "stem and progenitor cells",
    "multipotent": "stem and progenitor cells",
}
lineages = np.array(
    [next((v for k, v in LINEAGES.items() if k in t), "other") for t in cell_types]
)

model_2d = fit_autoencoder(DenoisingCountAutoencoder(NUM_GENES, 2), nb_loss)
codes_2d = {
    "PCA": PCA(2).fit(X_cells[train_cells]).transform(X_cells),
    "count autoencoder": encode_cells(model_2d),
}
plot_codes(
    {name: codes[test_cells] for name, codes in codes_2d.items()},
    lineages[test_cells],
    {name: cell_knn_accuracy(codes) for name, codes in codes_2d.items()},
    q=0.005,
)

The autoencoder keeps most of the information about cell types in 2 numbers, while PCA separates only the largest groups, mixing e.g. progenitors, granulocytic and dendritic cells. The differentiation of erythroid cells from progenitors forms a continuous branch. Plasma cells, never seen in training, are scattered: the encoder has no place for them.

#### Novelty detection

An autoencoder reconstructs well only data similar to its training data, so the reconstruction error is an **anomaly score**: plasma cells should get larger errors than known cell types. We use the MLP autoencoder with MSE loss, so that the errors are comparable with PCA, and AUROC as the metric (1 = perfect separation, 0.5 = random).

### Exercise 5: Anomaly score

Implement `reconstruction_error(model, x)`: the mean squared reconstruction error of every row of `x`, shape `(cells,)`.

In [ ]:
@torch.no_grad()
def reconstruction_error(model: nn.Module, x: torch.Tensor) -> torch.Tensor:
    ...

In [ ]:
x = X_cells_t[:5]
assert reconstruction_error(mlp_ae, x).shape == (5,)
assert torch.allclose(reconstruction_error(mlp_ae, x).mean(), F.mse_loss(mlp_ae(x), x))
assert (reconstruction_error(nn.Identity(), x) == 0).all()

pca_reconstruction = pca.inverse_transform(pca.transform(X_cells[test_cells]))
errors = {
    "PCA": np.mean((pca_reconstruction - X_cells[test_cells]) ** 2, axis=1),
    "MLP autoencoder": reconstruction_error(mlp_ae, X_cells_t[test_cells])
    .cpu()
    .numpy(),
}
novel = is_novel[test_cells]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, (name, error) in zip(axes, errors.items()):
    bins = np.linspace(0, np.quantile(error, 0.99), 40)
    ax.hist(error[~novel], bins, density=True, alpha=0.6, label="known cell types")
    ax.hist(error[novel], bins, density=True, alpha=0.6, label="plasma cells (novel)")
    auroc = roc_auc_score(novel, error)
    ax.set(title=f"{name}: AUROC {auroc:.3f}", xlabel="reconstruction error")
axes[0].legend()
plt.tight_layout()
plt.show()
print("OK")

PCA barely notices the new cell type, while the nonlinear autoencoder flags most plasma cells. The same approach detects e.g. fraud, machine failures or production defects, where anomalies are rare and diverse, so there is little to train a classifier on. A caveat: a too powerful autoencoder can reconstruct anomalies well, too.

# Part 2: Autoencoders for embeddings

Foundation models turn images, texts or molecules into **embeddings** that capture their meaning (Labs 1 and 2), so any data becomes tabular, and Part 1 applies. Their size is a practical problem: vector databases store millions or billions of them for similarity search. Compressing 384 numbers to 32 cuts storage and search time 12x, if the neighbors stay the same.

## 2.1 DINOv3 embeddings of Food-101

[DINOv3](https://arxiv.org/abs/2508.10104) (Meta, 2025) is a family of Vision Transformers trained with self-supervised learning on 1.7B unlabeled images, currently the best general-purpose image features. We use the smallest one, ViT-S/16 (21M parameters, 384-dimensional embeddings), distilled from a 7B model. Unlike the original on the HuggingFace Hub, the [timm](https://huggingface.co/docs/timm/index) copy doesn't require accepting the license (which still applies).

[Food-101](https://huggingface.co/datasets/ethz/food101) has 101k photos of 101 dishes, from pizza and sushi to beignets and pho, 750 training and 250 test images per class. The download is 4.8 GB; computing embeddings takes about 2 minutes on a GPU (up to 10 on Colab, where JPEG decoding on 2 CPU cores is the bottleneck), and they are cached on disk.

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader

# some Food-101 JPEGs have broken metadata, which is harmless
warnings.filterwarnings("ignore", message="Tag data larger than file")

food = load_dataset("ethz/food101")
food_classes = [n.replace("_", " ") for n in food["train"].features["label"].names]
y_food = {
    "train": np.array(food["train"]["label"]),
    "test": np.array(food["validation"]["label"]),
}
print(
    f"{len(food['train'])} train, {len(food['validation'])} test images, {len(food_classes)} classes"
)

DINO_MODEL = "vit_small_patch16_dinov3.lvd1689m"
dino = timm.create_model(DINO_MODEL, pretrained=True, num_classes=0).to(device).eval()
# DINOv3 preprocessing (ImageNet mean and std) at resolution 224
dino_config = timm.data.resolve_model_data_config(dino) | {
    "input_size": (3, 224, 224),
    "crop_pct": 1.0,
}
dino_transform = timm.data.create_transform(**dino_config)
print(
    f"DINOv3 ViT-S/16: {count_params(dino) / 1e6:.0f}M parameters, embedding size {dino.num_features}"
)

In [ ]:
def embed_images(dataset, model: nn.Module = dino) -> np.ndarray:
    # pooled embeddings (CLS token) of all images of a HuggingFace dataset
    dataset = dataset.with_transform(
        lambda batch: {
            "x": [dino_transform(image.convert("RGB")) for image in batch["image"]]
        }
    )
    loader = DataLoader(dataset, batch_size=128, num_workers=os.cpu_count())
    embeddings = []
    with torch.no_grad(), autocast():
        for batch in loader:
            embeddings.append(model(batch["x"].to(device)).float().cpu())
    return torch.cat(embeddings).numpy()


FOOD_PATH = os.path.join(DATA_ROOT, "food101_dinov3.npz")
if not os.path.exists(FOOD_PATH):
    start = time.time()
    np.savez(
        FOOD_PATH,
        train=embed_images(food["train"]),
        test=embed_images(food["validation"]),
    )
    print(f"embeddings computed in {time.time() - start:.0f} s")
E_food = dict(np.load(FOOD_PATH))
print(f"embeddings: {E_food['train'].shape} train, {E_food['test'].shape} test")

Similar dishes have similar embeddings. Below, test images and their nearest training images by cosine similarity:

In [ ]:
food_gallery = F.normalize(torch.tensor(E_food["train"], device=device), dim=1)

fig, axes = plt.subplots(3, 6, figsize=(14, 7.2))
for row, i in zip(axes, [10, 5000, 21000]):
    query = F.normalize(torch.tensor(E_food["test"][i], device=device), dim=0)
    neighbors = (food_gallery @ query).topk(5).indices.tolist()
    images = [food["validation"][i]] + [food["train"][j] for j in neighbors]
    for col, (ax, example) in enumerate(zip(row, images)):
        ax.imshow(example["image"].convert("RGB").resize((224, 224)))
        ax.set_title(
            ("query: " if col == 0 else "") + food_classes[example["label"]], fontsize=9
        )
        ax.axis("off")
plt.tight_layout()
plt.show()

## 2.2 Compressing embeddings

We compress the 384-dimensional embeddings to 32 numbers and measure:

* **kNN accuracy** - dish classification by 10 nearest training neighbors in the code space (cosine similarity); about 76% with the full embeddings
* **neighbor recall@10** - the fraction of the 10 nearest neighbors of a test image in the full embedding space that stay among its 10 nearest neighbors in the code space: does compression keep the search results of a vector database?
* **decompression** - a linear classifier trained on the original embeddings, applied to the decoded ones: can existing downstream models use the codes?

As in Part 1, inputs are standardized, with one scale for all dimensions to keep the geometry.

### Evaluation tools

In [ ]:
EMBEDDING_CODE_SIZE = 32
food_mean, food_scale = E_food["train"].mean(axis=0), E_food["train"].std(axis=0).mean()
Z_food = {
    s: torch.tensor((E - food_mean) / food_scale, device=device)
    for s, E in E_food.items()
}
E_food_t = {s: torch.tensor(E, device=device) for s, E in E_food.items()}
y_food_t = {s: torch.tensor(y, device=device) for s, y in y_food.items()}
embedding_results = {}


def accuracy(predictions: torch.Tensor, y: torch.Tensor) -> float:
    return (predictions == y).float().mean().item()


@torch.no_grad()
def nearest_neighbors(
    queries: torch.Tensor, gallery: torch.Tensor, k: int = 10
) -> torch.Tensor:
    # indices of the k most similar gallery vectors (cosine similarity)
    queries, gallery = F.normalize(queries, dim=1), F.normalize(gallery, dim=1)
    return torch.cat(
        [(q @ gallery.T).topk(k, dim=1).indices for q in queries.split(1000)]
    )


def food_knn_accuracy(codes: dict) -> float:
    # majority vote of 10 nearest training neighbors
    neighbors = nearest_neighbors(codes["test"], codes["train"])
    return accuracy(y_food_t["train"][neighbors].mode(dim=1).values, y_food_t["test"])


def train_linear_probe(X: torch.Tensor, y: torch.Tensor, epochs: int = 30) -> nn.Module:
    # multinomial logistic regression on the GPU, much faster than scikit-learn here
    torch.manual_seed(0)
    probe = nn.Linear(X.shape[1], int(y.max()) + 1).to(device)
    opt = torch.optim.AdamW(probe.parameters(), lr=1e-3, weight_decay=1e-4)
    for _ in range(epochs):
        for batch in torch.randperm(len(X), device=device).split(256):
            loss = F.cross_entropy(probe(X[batch]), y[batch])
            opt.zero_grad()
            loss.backward()
            opt.step()
    return probe.eval()


# reference values with the full, original embeddings (cosine similarity of the original
# embeddings defines the neighbors, as in a vector database)
true_neighbors = nearest_neighbors(E_food_t["test"][:2000], E_food_t["train"])
knn_accuracy = food_knn_accuracy(E_food_t)
full_probe = train_linear_probe(Z_food["train"], y_food_t["train"])
with torch.no_grad():
    probe_accuracy = accuracy(
        full_probe(Z_food["test"]).argmax(dim=1), y_food_t["test"]
    )
print(
    f"full embeddings: kNN accuracy {100 * knn_accuracy:.1f}%, linear probe {100 * probe_accuracy:.1f}%"
)


@torch.no_grad()
def evaluate_compression(name: str, encode: Callable, decode: Callable) -> None:
    codes = {split: encode(Z) for split, Z in Z_food.items()}
    found = nearest_neighbors(codes["test"][:2000], codes["train"])
    recall = (found[..., None] == true_neighbors[:, None]).any(dim=2).float().mean()
    probe = accuracy(full_probe(decode(codes["test"])).argmax(dim=1), y_food_t["test"])
    embedding_results[name] = {
        "kNN accuracy": food_knn_accuracy(codes),
        "neighbor recall@10": recall.item(),
        "probe on decoded embeddings": probe,
    }
    print(", ".join(f"{m}: {100 * v:.1f}%" for m, v in embedding_results[name].items()))
    title = f"Food-101, {EMBEDDING_CODE_SIZE}-dimensional codes"
    plot_results(embedding_results, title, xlim=(0, 100))


def fit_embedding_autoencoder(
    model: nn.Module, loss_fn: Callable, epochs: int = 20, lr: float = 1e-3
) -> nn.Module:
    # loss_fn(model, x) for a batch of standardized embeddings x
    transformers.set_seed(0)
    model = model.to(device)
    X = Z_food["train"]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps_per_epoch = len(X) // 512
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        opt, lr, total_steps=epochs * steps_per_epoch
    )
    start = time.time()
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X), device=device)[: steps_per_epoch * 512]
        for batch in order.split(512):
            loss = loss_fn(model, X[batch])
            opt.zero_grad()
            loss.backward()
            opt.step()
            scheduler.step()
    print(f"trained in {time.time() - start:.0f} s")
    return model.eval()

In [ ]:
pca_food = PCA(EMBEDDING_CODE_SIZE).fit(Z_food["train"].cpu().numpy())
pca_components = torch.tensor(pca_food.components_, device=device)
pca_center = torch.tensor(pca_food.mean_, device=device)
evaluate_compression(
    "1. PCA",
    encode=lambda Z: (Z - pca_center) @ pca_components.T,
    decode=lambda codes: codes @ pca_components + pca_center,
)

Now the MLP autoencoder from Part 1, with two hidden layers of 512, trained with `mse_loss`.

In [ ]:
food_ae = fit_embedding_autoencoder(
    MLPAutoencoder(384, EMBEDDING_CODE_SIZE, hidden=(512, 512)), mse_loss
)
evaluate_compression("2. MLP autoencoder", food_ae.encoder, food_ae.decoder)

The autoencoder wins in every metric, and its decoded embeddings work much better with the probe: PCA reconstructions lose a large part of the embedding, which makes them "atypical" for it.

### Exercise 6: Similarity-preserving loss

Retrieval needs not the exact reconstruction, but the **cosine similarities** between embeddings. We can preserve them directly, with a loss term comparing the cosine similarity matrices of a batch in the input and code spaces ([Matryoshka-Adaptor](https://arxiv.org/abs/2407.20243) uses a similar loss):

$$\mathcal{L} = \|x - \hat{x}\|^2 + \lambda \cdot \mathrm{MSE}\left(\cos(z_i, z_j), \cos(e_i, e_j)\right)$$

where $e$ are the original (not standardized) embeddings, $z$ the codes, and the MSE is averaged over all pairs $i, j$ in the batch. Implement `similarity_loss(model, x)`: compute the codes and the reconstruction, recover $e = x \cdot$ `food_scale` $+$ `food_mean`, and return the reconstruction MSE plus `weight` times the similarity term. Hint: for rows normalized with `F.normalize`, all pairwise cosine similarities are one matrix product.

In [ ]:
food_mean_t = torch.tensor(food_mean, device=device)


def similarity_loss(
    model: nn.Module, x: torch.Tensor, weight: float = 1.0
) -> torch.Tensor:
    ...

In [ ]:
class _IdentityAutoencoder(nn.Module):
    # encoder = decoder = identity: zero reconstruction loss, codes equal inputs
    def __init__(self) -> None:
        super().__init__()
        self.encoder = self.decoder = nn.Identity()


x = Z_food["train"][:64]
model = MLPAutoencoder(384, EMBEDDING_CODE_SIZE).to(device)
with torch.no_grad():
    assert torch.allclose(
        similarity_loss(model, x, weight=0.0), F.mse_loss(model(x), x)
    ), "with weight 0, only the reconstruction loss remains"
    # standardization changes cosine similarities, so even the identity has a positive loss
    assert similarity_loss(_IdentityAutoencoder(), x).item() > 1e-3, (
        "compare with cosine similarities of the original embeddings"
    )

food_ae_similarity = fit_embedding_autoencoder(
    MLPAutoencoder(384, EMBEDDING_CODE_SIZE, hidden=(512, 512)), similarity_loss
)
evaluate_compression(
    "3. + similarity loss", food_ae_similarity.encoder, food_ae_similarity.decoder
)
print("OK")

Neighbor recall improves by a few points, at almost the same accuracy. In a real system, the codes could be further quantized to 8 bits or binarized, for 50-400x compression of the database.

## 2.3 Visualization

A 2D map of 101 classes is hard to read, so we show 8 dishes. The 2-dimensional autoencoder is trained on all embeddings, like PCA.

In [ ]:
food_ae_2d = fit_embedding_autoencoder(
    MLPAutoencoder(384, 2, hidden=(512, 512)), mse_loss
)
dishes = np.array(food_classes)[y_food["test"]]
shown = np.isin(dishes, [
    "sushi", "pizza", "ramen", "ice cream", "french fries", "caesar salad", "steak", "macarons",
])  # fmt: skip
labels = dishes[shown]
with torch.no_grad():
    maps = {
        "PCA": PCA(2)
        .fit(Z_food["train"].cpu().numpy())
        .transform(Z_food["test"].cpu().numpy())[shown],
        "autoencoder": food_ae_2d.encoder(Z_food["test"]).cpu().numpy()[shown],
    }
# kNN accuracy of the 8 dishes from 2 numbers: train on even, test on odd test images
accuracies = {
    name: KNeighborsClassifier(10)
    .fit(xy[::2], labels[::2])
    .score(xy[1::2], labels[1::2])
    for name, xy in maps.items()
}
plot_codes(maps, labels, accuracies, q=0.01)

dino = food = food_gallery = None
free_memory()

PCA's two directions of the largest variance separate only a few groups. The autoencoder has to fit all 101 dishes into 2 numbers, and uses the space much better: most dishes form their own clusters.

# Part 3: Denoising autoencoders

A **denoising autoencoder** (DAE, [Vincent et al., 2008](https://dl.acm.org/doi/10.1145/1390156.1390294)) reconstructs the clean $x$ from a corrupted input $\tilde{x}$, e.g. with Gaussian noise added. Copying the input is useless, so no bottleneck is needed: the model must learn what clean data looks like, and "pull" the noisy input back to the data manifold. Besides regularization (Part 1), it's used for image restoration, and repeated denoising is the core of diffusion models.

The standard metric is **PSNR** (peak signal-to-noise ratio), $10 \log_{10}(1 / \mathrm{MSE})$ in decibels for images in $[0, 1]$. Higher is better; +3 dB means half the MSE, and 0.5 dB is already visible. PSNR measures pixel errors, not perceived quality, and a blurry image can score high ([Wang and Bovik, 2009](https://doi.org/10.1109/MSP.2008.930649)), so [SSIM](https://doi.org/10.1109/TIP.2003.819861) (structural similarity) or [LPIPS](https://arxiv.org/abs/1801.03924) (distance between deep network features) are often reported too.

## 3.1 Pretrained image denoiser

[DeepInverse](https://deepinv.github.io/) collects pretrained image restoration models. We use [DRUNet](https://arxiv.org/abs/2008.13751) (2021), a strong denoiser for natural images: a U-Net (Part 6) with residual blocks. It's **non-blind**: it gets the noise level $\sigma$ as an additional input channel, so one model handles all noise levels. **Blind** denoisers, like the Transformer-based [Restormer](https://arxiv.org/abs/2111.09881) (2022), estimate the noise level themselves.

In [ ]:
def psnr(x_hat: torch.Tensor, x: torch.Tensor) -> float:
    # mean PSNR of a batch of images in [0, 1]
    mse = ((x_hat.clamp(0, 1) - x) ** 2).flatten(1).mean(dim=1)
    return (10 * torch.log10(1 / mse)).mean().item()


drunet = deepinv.models.DRUNet(device=device).eval()
print(f"DRUNet: {count_params(drunet) / 1e6:.0f}M parameters")

photo = deepinv.utils.load_example("butterfly.png", img_size=256, device=device)
torch.manual_seed(0)
fig, axes = plt.subplots(2, 3, figsize=(10.5, 7.4))
for row, sigma in zip(axes, [25 / 255, 50 / 255]):
    noisy = photo + sigma * torch.randn_like(photo)
    with torch.no_grad():
        images = {"clean": photo, "noisy": noisy, "DRUNet": drunet(noisy, sigma)}
    for ax, (name, image) in zip(row, images.items()):
        ax.imshow(image[0].clamp(0, 1).permute(1, 2, 0).cpu())
        title = f"{name}, sigma {255 * sigma:.0f}/255" if name == "noisy" else name
        ax.set_title(
            title if name == "clean" else f"{title}: {psnr(image, photo):.1f} dB"
        )
        ax.axis("off")
plt.tight_layout()
plt.show()

DRUNet removes strong noise and keeps the edges, while simple filters blur everything. The $\sigma$ it gets controls the trade-off: too low leaves noise, too high removes details along with it. We will build this conditioning ourselves in Part 6.

## 3.2 Denoising CT scans

Computed tomography (CT) uses X-rays, so a lower dose is safer for patients, but gives noisier images. Denoising networks enable **low-dose CT** with diagnostic quality, and are used in clinical scanners. Real low-dose CT noise is complex; we simulate it with Gaussian noise, $\sigma = 0.1$.

We use 128 x 128 abdominal CT slices from [MedMNIST+](https://medmnist.com/) (OrganCMNIST, coronal view): 13k training and 8k test slices, labeled with one of 11 organs (used in Part 4). The baselines are a Gaussian blur and DRUNet, which works on CT despite being trained on photos: noise is noise.

In [ ]:
ct = np.load(download(
    "https://zenodo.org/records/10519652/files/organcmnist_128.npz?download=1",
    "organcmnist_128.npz",
))  # fmt: skip
ct_train = torch.tensor(ct["train_images"][:, None]).float().div(255)
ct_test = torch.tensor(ct["test_images"][:, None]).float().div(255)
ct_labels = {
    split: torch.tensor(ct[f"{split}_labels"][:, 0]) for split in ("train", "test")
}
print(f"CT slices: {tuple(ct_train.shape)} train, {tuple(ct_test.shape)} test")


def fixed_noise(shape: torch.Size, seed: int) -> torch.Tensor:
    # standard Gaussian noise, the same on every device
    return torch.randn(shape, generator=torch.Generator().manual_seed(seed)).to(device)


CT_SIGMA = 0.1
# fixed noisy test set: the first 500 test slices
ct_clean = ct_test[:500].to(device)
ct_noisy = ct_clean + CT_SIGMA * fixed_noise(ct_clean.shape, seed=0)

gray_drunet = deepinv.models.DRUNet(in_channels=1, out_channels=1, device=device).eval()
with torch.no_grad():
    ct_drunet = torch.cat(
        [gray_drunet(batch, CT_SIGMA) for batch in ct_noisy.split(50)]
    )
gray_drunet = drunet = None
free_memory()

ct_baselines = {
    "noisy": ct_noisy,
    "Gaussian blur": torchvision.transforms.functional.gaussian_blur(ct_noisy, 5, 1.0),
    "DRUNet (pretrained, photos)": ct_drunet,
}
for name, images in ct_baselines.items():
    print(f"{name}: {psnr(images, ct_clean):.2f} dB")

### Upsampling

A convolutional autoencoder downsamples with strided convolutions or pooling, so the decoder needs **upsampling** layers. The three common options from the lecture:

* **transposed convolution** - every input pixel "paints" a scaled copy of the learned kernel, with stride $s$; if the kernel size isn't divisible by the stride, the copies overlap unevenly, giving **checkerboard artifacts**. A $2 \times 2$ kernel with stride 2 has no overlap
* **interpolation + convolution** - nearest neighbor upsampling, then a $3 \times 3$ convolution; no artifacts, but the convolution runs at the higher resolution
* **pixel shuffle** - a convolution at low resolution to $4C$ channels, rearranged into a 2x larger map with $C$ channels; cheap and without artifacts

### Exercise 7: Upsampling layers

Implement `upsample(kind, in_channels, out_channels)`, returning a module that doubles the spatial size:

* `"transposed"`: [`nn.ConvTranspose2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html) with kernel size 2 and stride 2
* `"transposed_3x3"`: transposed convolution with kernel size 3, stride 2, `padding=1` and `output_padding=1` (the checkerboard-prone version)
* `"interpolation"`: [`nn.Upsample`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html) (`scale_factor=2`, nearest) and a 3x3 convolution with `padding=1`
* `"pixel_shuffle"`: a 3x3 convolution with `padding=1` to `4 * out_channels` channels and [`nn.PixelShuffle(2)`](https://docs.pytorch.org/docs/stable/generated/torch.nn.PixelShuffle.html)

In [ ]:
def upsample(kind: str, in_channels: int, out_channels: int) -> nn.Module:
    ...
    raise ValueError(f"unknown upsampling: {kind}")

In [ ]:
UPSAMPLINGS = ["transposed", "transposed_3x3", "interpolation", "pixel_shuffle"]
for kind in UPSAMPLINGS:
    layer = upsample(kind, 8, 4)
    assert layer(torch.randn(2, 8, 5, 7)).shape == (2, 4, 10, 14), (
        f"{kind}: wrong output shape"
    )

# 3 upsampling layers with random weights (positive, as after ReLUs), applied to a constant image
torch.manual_seed(0)
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, kind in zip(axes, UPSAMPLINGS):
    layers = nn.Sequential(*[upsample(kind, 8, 8) for _ in range(3)])
    for p in layers.parameters():
        nn.init.uniform_(p, 0, 1)
    with torch.no_grad():
        image = layers(torch.ones(1, 8, 4, 4))[0].mean(dim=0)
    ax.imshow(image, cmap="gray")
    ax.set_title(kind)
    ax.axis("off")
plt.suptitle("Constant input, 3 upsampling layers with random positive weights")
plt.tight_layout()
plt.show()
print("OK")

The 3x3 transposed convolution gives a strong grid pattern, which the network must learn to cancel. The 2x2 transposed convolution and pixel shuffle repeat one pattern in every 2x2 block, which is easy to correct. Interpolation is smooth by design, at the cost of convolving at high resolution.

### Exercise 8: Convolutional denoising autoencoder

Implement `ConvDAE`, a fully convolutional autoencoder, with the given `conv_block(cin, cout)` (two 3x3 convolutions with BatchNorm and ReLU, as in U-Net):

* encoder: for every width in `widths`, a `conv_block` (from 1 input channel for the first one) and 2x2 max pooling; with `widths=(64, 128)`, $128 \times 128$ images become $32 \times 32$ maps with 128 channels
* decoder: mirrors it, for every width in reversed order `upsample(...)` to that width and a `conv_block`; then a 1x1 convolution to 1 channel

The bottleneck isn't narrow ($32 \times 32 \times 128$ is 8x more numbers than input pixels), but noise makes the identity useless anyway. Deeper encoders, with 3 or 4 poolings, lost too much spatial detail here.

In [ ]:
def conv_block(in_channels: int, out_channels: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


class ConvDAE(nn.Module):
    def __init__(
        self, widths: tuple[int, ...] = (64, 128), upsampling: str = "interpolation"
    ) -> None:
        super().__init__()
        ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))

In [ ]:
model = ConvDAE()
x = torch.randn(2, 1, 128, 128)
assert model.encoder(x).shape == (2, 128, 32, 32), (
    f"got {tuple(model.encoder(x).shape)}"
)
assert model(x).shape == (2, 1, 128, 128)
assert count_params(model) == 850241, (
    f"expected 850241 parameters, got {count_params(model)}"
)
print("OK")

`fit_denoiser` trains a denoiser on random batches of training slices with fresh noise, sampled by the `noise` function (by default Gaussian, $\sigma = 0.1$). Training takes about a minute on a GPU.

In [ ]:
DENOISING_STEPS = 2000
ct_train_gpu = ct_train.to(device)


def gaussian_noise(x: torch.Tensor) -> tuple[torch.Tensor, dict]:
    # returns the noisy input and additional arguments of the model
    return x + CT_SIGMA * torch.randn_like(x), {}


def fit_denoiser(
    model: nn.Module, noise: Callable = gaussian_noise, steps: int = DENOISING_STEPS
) -> nn.Module:
    transformers.set_seed(0)
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
    start = time.time()
    for step in range(steps):
        model.train()
        x = ct_train_gpu[torch.randint(len(ct_train_gpu), (32,), device=device)]
        noisy, kwargs = noise(x)
        with autocast():
            loss = F.mse_loss(model(noisy, **kwargs).float(), x)
        opt.zero_grad()
        loss.backward()
        opt.step()
        scheduler.step()
    print(f"trained in {time.time() - start:.0f} s")
    return model.eval()


@torch.no_grad()
def denoise(
    model: nn.Module, noisy: torch.Tensor, sigma: float | None = None
) -> torch.Tensor:
    # in batches; sigma is passed only to noise-conditioned models
    outputs = []
    with autocast():
        for batch in noisy.split(100):
            kwargs = (
                {} if sigma is None else {"sigma": batch.new_full((len(batch),), sigma)}
            )
            outputs.append(model(batch, **kwargs).float())
    return torch.cat(outputs)


conv_dae = fit_denoiser(ConvDAE())
ct_denoised = {**ct_baselines, "conv DAE": denoise(conv_dae, ct_noisy)}
for name, images in ct_denoised.items():
    print(f"{name}: {psnr(images, ct_clean):.2f} dB")

In [ ]:
def show_ct(images: dict, indices: list[int], crop: slice = slice(None)) -> None:
    # rows: test slices, columns: methods
    fig, axes = plt.subplots(
        len(indices), len(images), figsize=(2.6 * len(images), 2.7 * len(indices))
    )
    for row, i in zip(np.atleast_2d(axes), indices):
        for ax, (name, batch) in zip(row, images.items()):
            ax.imshow(
                batch[i, 0, crop, crop].float().cpu().clamp(0, 1),
                cmap="gray",
                vmin=0,
                vmax=1,
            )
            ax.set_title(
                name
                if name == "clean"
                else f"{name}\n{psnr(batch[i : i + 1], ct_clean[i : i + 1]):.1f} dB",
                fontsize=9,
            )
            ax.axis("off")
    plt.tight_layout()
    plt.show()


show_ct({"clean": ct_clean, **ct_denoised}, [3, 11])

The conv DAE removes the noise better than blurring, but smooths out thin edges and small structures: everything passes through the 4x downsampled bottleneck, and the decoder must "invent" the details from coarse features. DRUNet, a U-Net, is better despite never seeing a CT scan: its skip connections pass the details around the bottleneck. We will build one in Part 6.

# Part 4: Masked autoencoders

A **masked autoencoder** (MAE, [He et al., 2022](https://arxiv.org/abs/2111.06377)) is a denoising autoencoder whose corruption removes a random 75% of image patches; the model reconstructs the missing pixels. Images are very redundant, but with so much masked, copying neighbors isn't enough: the model has to understand the image. The architecture is asymmetric:

* the **encoder**, a ViT, sees only the visible patches, so pretraining is ~3x cheaper
* a small **decoder** gets the encoded patches plus a learned **mask token** for every missing one (with positional embeddings), and predicts their pixels
* the loss is the MSE on the **masked patches only**, with pixels normalized per patch

After pretraining, the decoder is discarded and the encoder is fine-tuned for downstream tasks. It's the image version of BERT's masked language modeling (Lab 2), also used in ConvNeXt V2 (FCMAE, with sparse convolutions), VideoMAE (90% masking, since videos are even more redundant) and many models for medicine, satellite images or audio.

## 4.1 Pretrained MAE

[`facebook/vit-mae-base`](https://huggingface.co/facebook/vit-mae-base) is the original MAE ViT-B/16 pretrained on ImageNet, with the decoder. `ViTMAEForPreTraining` returns the predicted pixels of all patches (ImageNet-normalized) and the mask (1 = removed); the `noise` argument fixes which patches are masked, those with the largest values. We reconstruct Food-101 test images at increasing masking ratios.

In [ ]:
from transformers import AutoImageProcessor, ViTMAEForPreTraining

mae_processor = AutoImageProcessor.from_pretrained("facebook/vit-mae-base")
mae = ViTMAEForPreTraining.from_pretrained("facebook/vit-mae-base").to(device).eval()
imagenet_mean = torch.tensor(mae_processor.image_mean, device=device).view(3, 1, 1)
imagenet_std = torch.tensor(mae_processor.image_std, device=device).view(3, 1, 1)

food_test = load_dataset("ethz/food101", split="validation")
images = [food_test[i]["image"].convert("RGB") for i in [0, 15000]]
pixels = mae_processor(images=images, return_tensors="pt")["pixel_values"].to(device)
noise = torch.rand(len(images), 196, generator=torch.Generator().manual_seed(0))


def to_image(x: torch.Tensor) -> torch.Tensor:
    # ImageNet-normalized (3, H, W) -> (H, W, 3) in [0, 1]
    return (x * imagenet_std + imagenet_mean).clamp(0, 1).permute(1, 2, 0).cpu()


ratios = [0.5, 0.75, 0.9]
fig, axes = plt.subplots(len(images), 1 + 2 * len(ratios), figsize=(16, 5.2))
for j, ratio in enumerate(ratios):
    mae.config.mask_ratio = ratio
    with torch.no_grad():
        out = mae(pixel_values=pixels, noise=noise.to(device))
    reconstruction = mae.unpatchify(out.logits)
    mask = mae.unpatchify(out.mask[..., None].repeat(1, 1, 16 * 16 * 3))  # 1 = masked
    # visible patches pasted back, only the masked ones are predicted
    filled = reconstruction * mask + pixels * (1 - mask)
    for i, row in enumerate(axes):
        row[0].imshow(to_image(pixels[i]))
        row[0].set_title("original", fontsize=9)
        row[1 + 2 * j].imshow(to_image(pixels[i] * (1 - mask[i])))
        row[1 + 2 * j].set_title(f"{100 * ratio:.0f}% masked", fontsize=9)
        row[2 + 2 * j].imshow(to_image(filled[i]))
        row[2 + 2 * j].set_title("reconstruction", fontsize=9)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

mae = food_test = None
free_memory()

Even with 90% of the image removed, the model recovers the overall layout (plate, food, background) with plausible colors. The reconstructions are blurry, since MSE rewards the average of all plausible fillings; that's fine, as the goal is the representation, not image generation.

## 4.2 MAE pretraining for few labels

MAE needs no labels, which suits medicine: images are plentiful, expert annotations are not. We pretrain a small ViT with MAE on the CT slices from Part 3, and fine-tune it for **organ classification** (11 organs) with only 1% or 10% of the labels, compared to the same ViT trained from scratch.

For speed: 64 x 64 slices and 8 x 8 patches (64 per image), a ViT with 6 blocks of width 192 (as ViT-Tiny; `Block` from timm is the Transformer block from Lab 2), and a decoder with 2 blocks of width 128.

### Exercise 9: Random masking

Implement `random_masking(batch_size, num_patches, mask_ratio)`, with an independent random mask for every image:

1. sort uniform noise of shape `(batch_size, num_patches)` ([`argsort`](https://docs.pytorch.org/docs/stable/generated/torch.argsort.html)); the first `num_keep = int(num_patches * (1 - mask_ratio))` indices of every row are the visible patches, `keep` with shape `(batch_size, num_keep)`
2. the boolean `mask` of shape `(batch_size, num_patches)` is `True` for masked patches: start with all `True` and set `False` at `keep`, e.g. with [`scatter`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.scatter_.html)

In [ ]:
def random_masking(
    batch_size: int, num_patches: int, mask_ratio: float, device: torch.device = device
) -> tuple[torch.Tensor, torch.Tensor]:
    ...
    return keep, mask

In [ ]:
keep, mask = random_masking(500, 64, 0.75, torch.device("cpu"))
assert keep.shape == (500, 16) and mask.shape == (500, 64) and mask.dtype == torch.bool
assert (mask.sum(dim=1) == 48).all(), "75% of the patches should be masked"
assert not mask.gather(1, keep).any(), "the kept patches should not be masked"
assert all(len(set(row.tolist())) == 16 for row in keep), (
    "kept patches should be different"
)
# every patch is masked with the same probability, and masks differ between images
assert 0.65 < mask.float().mean(dim=0).min() and mask.float().mean(dim=0).max() < 0.85
assert not (mask[0] == mask[1]).all()
print("OK")

### Exercise 10: MAE

Complete `MaskedAutoencoder.forward`, which returns the loss for images `x` with shape `(B, 1, 64, 64)`. The encoder (`ViTEncoder`, which takes the indices of visible patches) and the decoder layers are given.

1. sample the mask with `random_masking`, and encode the visible patches: `(B, num_keep, dim)`
2. project them with `self.to_decoder`, and build the decoder input `(B, num_patches, decoder_dim)`: the mask token everywhere, with the encoded patches put back at positions `keep` (`scatter` along dimension 1, with indices expanded to the decoder width); add `self.decoder_positions`
3. run the decoder blocks and `self.decoder_norm`, and predict the pixels of every patch with `self.predict`: `(B, num_patches, patch_size ** 2)`
4. the target: patches of `x` from `patchify`, `(B, num_patches, patch_size ** 2)`, each normalized to zero mean and unit standard deviation (+ `1e-6`)
5. return the MSE averaged over the masked patches only

In [ ]:
from timm.models.vision_transformer import Block

MAE_RESOLUTION, MAE_PATCH = 64, 8
NUM_PATCHES = (MAE_RESOLUTION // MAE_PATCH) ** 2


def patchify(x: torch.Tensor) -> torch.Tensor:
    # (B, 1, H, W) -> (B, num_patches, patch_size ** 2), patches in row-major order
    return F.unfold(x, MAE_PATCH, stride=MAE_PATCH).transpose(1, 2)


class ViTEncoder(nn.Module):
    def __init__(self, dim: int = 192, depth: int = 6, heads: int = 3) -> None:
        super().__init__()
        self.patch_embedding = nn.Conv2d(1, dim, MAE_PATCH, stride=MAE_PATCH)
        self.positions = nn.Parameter(0.02 * torch.randn(1, NUM_PATCHES, dim))
        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)

    def forward(
        self, x: torch.Tensor, keep: torch.Tensor | None = None
    ) -> torch.Tensor:
        # (B, 1, H, W) -> (B, num_patches, dim), or only the patches in keep (B, num_keep)
        tokens = self.patch_embedding(x).flatten(2).transpose(1, 2) + self.positions
        if keep is not None:
            tokens = tokens.gather(1, keep[..., None].expand(-1, -1, tokens.shape[-1]))
        for block in self.blocks:
            tokens = block(tokens)
        return self.norm(tokens)


class MaskedAutoencoder(nn.Module):
    def __init__(self, mask_ratio: float = 0.75, decoder_dim: int = 128) -> None:
        super().__init__()
        self.mask_ratio = mask_ratio
        self.encoder = ViTEncoder()
        self.to_decoder = nn.Linear(192, decoder_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_dim))
        self.decoder_positions = nn.Parameter(
            0.02 * torch.randn(1, NUM_PATCHES, decoder_dim)
        )
        self.decoder_blocks = nn.ModuleList([Block(decoder_dim, 4) for _ in range(2)])
        self.decoder_norm = nn.LayerNorm(decoder_dim)
        self.predict = nn.Linear(decoder_dim, MAE_PATCH**2)

    def forward(self, x: torch.Tensor, return_prediction: bool = False):
        B = len(x)
        ...
        if return_prediction:
            return loss, prediction, mask
        return loss

In [ ]:
torch.manual_seed(0)
model = MaskedAutoencoder()
x = torch.rand(4, 1, MAE_RESOLUTION, MAE_RESOLUTION)
loss, prediction, mask = model(x, return_prediction=True)
assert loss.ndim == 0 and prediction.shape == (4, NUM_PATCHES, MAE_PATCH**2)
assert (mask.sum(dim=1) == 48).all()
# the loss uses only the masked patches: changing the prediction for visible ones changes nothing
with torch.no_grad():
    model.predict.bias.zero_()
    target = patchify(x)
    target = (target - target.mean(dim=-1, keepdim=True)) / (
        target.std(dim=-1, keepdim=True) + 1e-6
    )
    torch.manual_seed(1)
    loss, prediction, mask = model(x, return_prediction=True)
    expected = ((prediction - target) ** 2).mean(dim=-1)[mask].mean()
assert torch.allclose(loss, expected, atol=1e-5), (
    "average the MSE over masked patches only, with normalized targets"
)
print(
    f"MAE: encoder {count_params(model.encoder) / 1e6:.1f}M, whole model {count_params(model) / 1e6:.1f}M parameters"
)
print("OK")

Pretraining for 100 epochs on 13k unlabeled slices takes about a minute on a modern GPU (4-5 minutes on Colab T4).

In [ ]:
ct_small = {
    "train": F.interpolate(ct_train, size=MAE_RESOLUTION, mode="area").to(device),
    "test": F.interpolate(ct_test, size=MAE_RESOLUTION, mode="area").to(device),
}
ct_y = {split: y.to(device) for split, y in ct_labels.items()}


def fit_vit(
    model: nn.Module, loss_fn: Callable, steps: int, lr: float, batch_size: int = 256
) -> nn.Module:
    # AdamW with warmup and cosine decay, as in MAE; loss_fn(model, batch_size) -> loss
    transformers.set_seed(0)
    model = model.to(device)
    opt = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=0.05, betas=(0.9, 0.95)
    )
    scheduler = transformers.get_cosine_schedule_with_warmup(opt, steps // 10, steps)
    for step in range(steps):
        model.train()
        with autocast():
            loss = loss_fn(model, batch_size)
        opt.zero_grad()
        loss.backward()
        opt.step()
        scheduler.step()
    return model.eval()


def mae_loss(model: nn.Module, batch_size: int) -> torch.Tensor:
    x = ct_small["train"]
    return model(x[torch.randint(len(x), (batch_size,), device=device)])


start = time.time()
MAE_STEPS = 100 * len(ct_small["train"]) // 256  # 100 epochs
pretrained_mae = fit_vit(MaskedAutoencoder(), mae_loss, steps=MAE_STEPS, lr=1.5e-3)
print(f"MAE pretraining: {time.time() - start:.0f} s")

# reconstructions of test slices; predictions are normalized per patch, so we
# use the mean and std of the true patch to show them
with torch.no_grad():
    x = ct_small["test"][:6]
    _, prediction, mask = pretrained_mae(x, return_prediction=True)
patches = patchify(x)
mean, std = patches.mean(dim=-1, keepdim=True), patches.std(dim=-1, keepdim=True)
filled = torch.where(mask[..., None], prediction.float() * std + mean, patches)


def fold(p: torch.Tensor) -> torch.Tensor:
    return F.fold(p.transpose(1, 2), MAE_RESOLUTION, MAE_PATCH, stride=MAE_PATCH)


rows = {
    "original": x,
    "75% masked": fold(patches * ~mask[..., None]),
    "reconstruction": fold(filled),
}
fig, axes = plt.subplots(3, 6, figsize=(12, 6.4))
for row, (name, images) in zip(axes, rows.items()):
    for ax, image in zip(row, images):
        ax.imshow(image[0].cpu().clamp(0, 1), cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
    row[0].set_title(name, loc="left")
plt.tight_layout()
plt.show()

The visualization uses the true mean and standard deviation of every patch, so the reconstructions look better than they are; the structure inside masked patches is predicted, though. Now fine-tuning: the encoder (pretrained or random) with a linear classifier on mean-pooled patch tokens, trained on 1% or 10% of the labeled slices, 10-20 seconds per run.

In [ ]:
import copy


class ViTClassifier(nn.Module):
    def __init__(self, encoder: ViTEncoder, num_classes: int = 11) -> None:
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(192, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(x).mean(dim=1))


def finetune_accuracy(
    encoder: ViTEncoder, labeled: torch.Tensor, steps: int = 1000
) -> float:
    def loss_fn(model: nn.Module, bs: int) -> torch.Tensor:
        batch = labeled[
            torch.randint(len(labeled), (min(bs, len(labeled)),), device=device)
        ]
        logits = model(ct_small["train"][batch]).float()
        return F.cross_entropy(logits, ct_y["train"][batch], label_smoothing=0.1)

    model = fit_vit(
        ViTClassifier(encoder), loss_fn, steps=steps, lr=1e-3, batch_size=128
    )
    with torch.no_grad(), autocast():
        predictions = torch.cat(
            [model(x).argmax(dim=1) for x in ct_small["test"].split(1000)]
        )
    return (predictions == ct_y["test"]).float().mean().item()


mae_results = {}
for fraction in [0.01, 0.1]:
    labeled = torch.tensor(
        rng.permutation(len(ct_y["train"]))[: int(fraction * len(ct_y["train"]))],
        device=device,
    )
    for name, encoder in [
        ("from scratch", ViTEncoder()),
        ("MAE pretrained", copy.deepcopy(pretrained_mae.encoder)),
    ]:
        mae_results.setdefault(
            f"{100 * fraction:.0f}% labels ({len(labeled)} slices)", {}
        )[name] = finetune_accuracy(encoder, labeled)
print(pd.DataFrame(mae_results).T.map(lambda a: f"{100 * a:.1f}%"))

MAE pretraining improves the accuracy by 15-25 points with few labels: the encoder already knows the anatomy and only learns which organ is which, while training from scratch on 130 slices mostly overfits. With all labels, the gap shrinks to about 3 points in our experiments. Medical foundation models are pretrained this way on millions of scans.

# Part 5: Sparse autoencoders

Can we understand what a neural network computes inside? Single neurons are usually **polysemantic**, firing for several unrelated concepts: networks represent more concepts than they have neurons, as nearly orthogonal directions in the activation space (**superposition**, [Elhage et al., 2022](https://transformer-circuits.pub/2022/toy_model/index.html)).

A **sparse autoencoder** (SAE) decomposes an activation vector $x \in \mathbb{R}^d$ into a sparse combination of many learned directions:

$$f = \sigma(W_{enc}(x - b_{dec}) + b_{enc}) \in \mathbb{R}^{m}, \quad \hat{x} = W_{dec}^\top f + b_{dec} = \sum_i f_i d_i + b_{dec}, \quad m \gg d$$

The latent is **wide** (e.g. $m = 16d$) but **sparse**: only a few features are active for any input. Every feature $i$ has an encoder row (detecting the concept) and a unit-norm decoder row $d_i$ (its direction). Sparse features tend to be **monosemantic**. Sparsity is enforced with:

* **ReLU + L1 penalty** ([Bricken et al., 2023](https://transformer-circuits.pub/2023/monosemantic-features)) - simple, but L1 also shrinks the active features toward zero
* **TopK** ([Gao et al., 2024](https://arxiv.org/abs/2406.04093)) - keep the $k$ largest pre-activations and zero the rest; sparsity is set directly, no shrinkage
* **JumpReLU** ([Rajamanoharan et al., 2024](https://arxiv.org/abs/2407.14435)) - zero below a learned per-feature threshold $\theta_i$, keep the value above it: $f_i = z_i \cdot [z_i > \theta_i]$

## 5.1 Interpreting and steering an LLM

[Gemma Scope 2](https://huggingface.co/google/gemma-scope-2-1b-it) (Google DeepMind, 2025) is a set of JumpReLU SAEs trained on activations of every layer of the Gemma 3 models. We use the instruction-tuned [Gemma 3 1B](https://huggingface.co/unsloth/gemma-3-1b-it) (an ungated copy) and the SAE with 16k features on the **residual stream** after layer 13 of 26, i.e. the output of the 13th Transformer block. Gemma overflows in float16 (Lab 2), so GPUs without bfloat16 (e.g. Colab T4) use float32, about 4 GB.

[Neuronpedia](https://www.neuronpedia.org/) hosts explanations of SAE features, generated by an LLM from their top-activating texts, with a free API.

In [ ]:
import requests
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoModelForCausalLM, AutoTokenizer

GEMMA_MODEL = "unsloth/gemma-3-1b-it"
SAE_LAYER = 13
gemma_dtype = torch.bfloat16 if HALF_DTYPE == torch.bfloat16 else torch.float32
gemma_tokenizer = AutoTokenizer.from_pretrained(GEMMA_MODEL)
gemma = (
    AutoModelForCausalLM.from_pretrained(GEMMA_MODEL, dtype=gemma_dtype)
    .to(device)
    .eval()
)

# download only the parameters of one SAE: the whole repository has terabytes
sae_path = hf_hub_download(
    "google/gemma-scope-2-1b-it",
    f"resid_post/layer_{SAE_LAYER}_width_16k_l0_medium/params.safetensors",
)
gemma_sae = {name: p.to(device) for name, p in load_file(sae_path).items()}
print({name: tuple(p.shape) for name, p in gemma_sae.items()})


def feature_label(index: int) -> str:
    # explanation of a Gemma Scope feature from the Neuronpedia API (cached)
    if index not in feature_label.cache:
        url = f"https://www.neuronpedia.org/api/feature/gemma-3-1b-it/{SAE_LAYER}-gemmascope-2-res-16k/{index}"
        try:
            explanations = requests.get(url, timeout=10).json().get("explanations", [])
            feature_label.cache[index] = (
                explanations[0]["description"] if explanations else "(no explanation)"
            )
        except (requests.RequestException, ValueError):
            feature_label.cache[index] = "(Neuronpedia unavailable)"
    return feature_label.cache[index]


feature_label.cache = {}

### Exercise 11: JumpReLU encoder

Gemma Scope stores `w_enc` with shape `(d, m)` and `w_dec` with shape `(m, d)`, and doesn't subtract `b_dec` before encoding. Implement:

* `sae_encode(x)`: pre-activations $z = x W_{enc} + b_{enc}$, then JumpReLU with the per-feature `threshold`
* `sae_decode(f)`: $\hat{x} = f W_{dec} + b_{dec}$

`residual_stream(text)` (given) returns the residual stream after layer 13, `(tokens, d)`, captured with a **forward hook**: a function that PyTorch calls after the forward pass of a module, here a Transformer block.

In [ ]:
def sae_encode(x: torch.Tensor) -> torch.Tensor:
    ...


def sae_decode(f: torch.Tensor) -> torch.Tensor:
    ...


@torch.no_grad()
def residual_stream(text: str) -> tuple[torch.Tensor, list[str]]:
    captured = {}

    def hook(module, inputs, output):
        captured["x"] = output[0] if isinstance(output, tuple) else output

    handle = gemma.model.layers[SAE_LAYER].register_forward_hook(hook)
    inputs = gemma_tokenizer(text, return_tensors="pt").to(device)
    gemma(**inputs)
    handle.remove()
    tokens = gemma_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    return captured["x"][0].float(), tokens

In [ ]:
x, tokens = residual_stream(
    "My dog loves coffee, and at night we look at the stars through a telescope."
)
f = sae_encode(x)
assert f.shape == (len(tokens), 16384) and (f >= 0).all()
l0 = (f > 0).sum(dim=1).float()[1:]  # without <bos>, which has a huge norm
fvu = ((sae_decode(f) - x)[1:] ** 2).sum() / ((x[1:] - x[1:].mean(dim=0)) ** 2).sum()
print(
    f"active features per token: {l0.mean():.0f} of 16384, fraction of variance unexplained: {fvu:.3f}"
)
assert 20 < l0.mean() < 150 and fvu < 0.3, "check the encoder and decoder"

for token, row in list(zip(tokens, f))[1:]:
    values, indices = row.topk(2)
    described = ", ".join(
        f"#{i} {feature_label(int(i))!r} ({v:.0f})" for v, i in zip(values, indices)
    )
    print(f"{token:>12s}: {described}")
print("OK")

About 60 of 16k features are active per token, reconstructing about 75% of the variance. Many top features are readable: they fire on the word itself ("dogs and pets" on "dog"), its role ("liking and enjoyment" on "loves"), or the context. Some are vague or unexplained: automatic interpretation is far from perfect.

### Steering

Features are causal: adding a multiple of a feature's decoder direction $d_i$ to the residual stream makes the model "think" about the concept. We use three features found with the Neuronpedia search: "coffee, drinks, and related items" (4667), "stars and space" (13981) and "dogs and pets" (8908). Gemma's residual stream has a norm of thousands, hence the large coefficients.

### Exercise 12: Steering hook

Implement `steering_hook(direction, coefficient)`, which returns a forward hook adding `coefficient * direction` (in the dtype of the hidden states) to the output of a Transformer block. If the block returns a tuple, the hidden states are first: return a tuple with them modified and the other elements unchanged.

In [ ]:
def steering_hook(direction: torch.Tensor, coefficient: float) -> Callable:
    ...


def chat(
    question: str,
    feature: int | None = None,
    coefficient: float = 0.0,
    max_new_tokens: int = 60,
) -> str:
    handle = None
    if feature is not None:
        hook = steering_hook(gemma_sae["w_dec"][feature], coefficient)
        handle = gemma.model.layers[SAE_LAYER].register_forward_hook(hook)
    inputs = gemma_tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(device)
    try:
        output = gemma.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False
        )
    finally:
        if handle is not None:
            handle.remove()
    return gemma_tokenizer.decode(
        output[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True
    ).strip()

In [ ]:
# the hook adds the direction to the hidden states, in both output formats
direction = torch.randn(8)
hidden = torch.randn(2, 3, 8)
out = steering_hook(direction, 2.0)(None, None, (hidden, "cache"))
assert (
    isinstance(out, tuple)
    and out[1] == "cache"
    and torch.allclose(out[0], hidden + 2 * direction)
)
assert torch.allclose(
    steering_hook(direction, -1.0)(None, None, hidden), hidden - direction
)

question = "Describe your perfect weekend in two sentences."
print(f"QUESTION: {question}\n\nNO STEERING: {chat(question)}\n")
for feature, coefficients in [
    (4667, [200, 400, 800]),
    (13981, [200, 400, 800]),
    (8908, [400, 800]),
]:
    for coefficient in coefficients:
        print(
            f"'{feature_label(feature)}' x {coefficient}: {chat(question, feature, coefficient)}\n"
        )
print("OK")

With a moderate coefficient, the concept appears naturally in a sensible answer; too strong, and the model produces gibberish, as the residual stream is pushed far from its training activations. Some features steer weakly ("dogs and pets" barely changes the answer): a feature that **detects** a concept doesn't always **cause** it. Steering is used to control style, make models safer, and remove unwanted behaviors, e.g. by ablating the "refusal direction" ([Arditi et al., 2024](https://arxiv.org/abs/2406.11717)).

In [ ]:
gemma = gemma_sae = None
free_memory()

## 5.2 Our own SAE: concepts in DINOv3 embeddings

Now we train an SAE ourselves, on the standardized DINOv3 embeddings of Food-101 from Part 2. The labels give only the dish, but the embeddings encode much more: ingredients, colors, shapes, plating. With 4096 features (about 10x the embedding size) and 75k training images, it trains in seconds.

### Exercise 13: TopK SAE

Implement `TopKSAE.encode`: compute the pre-activations $z = (x - b_{dec}) W_{enc} + b_{enc}$, keep the `k` largest values of every row ([`topk`](https://docs.pytorch.org/docs/stable/generated/torch.topk.html)) passed through ReLU, and zero the rest (e.g. `torch.zeros_like(z).scatter(...)`). `W_enc` has shape `(d, m)` and `W_dec` `(m, d)`. A ReLU SAE with L1 penalty is given for comparison.

In [ ]:
class ReLUSAE(nn.Module):
    def __init__(self, dim: int, num_features: int) -> None:
        super().__init__()
        # decoder rows: unit-norm feature directions; encoder initialized as their transpose
        self.W_dec = nn.Parameter(F.normalize(torch.randn(num_features, dim), dim=1))
        self.W_enc = nn.Parameter(self.W_dec.data.T.clone())
        self.b_enc = nn.Parameter(torch.zeros(num_features))
        self.b_dec = nn.Parameter(torch.zeros(dim))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return F.relu((x - self.b_dec) @ self.W_enc + self.b_enc)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        f = self.encode(x)
        return f @ self.W_dec + self.b_dec, f


class TopKSAE(ReLUSAE):
    def __init__(self, dim: int, num_features: int, k: int) -> None:
        super().__init__(dim, num_features)
        self.k = k

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        ...

In [ ]:
sae = TopKSAE(6, 10, k=3)
with torch.no_grad():
    sae.W_enc.copy_(torch.eye(6, 10))  # the first 6 features copy the input
    x = torch.tensor([[5.0, -1.0, 3.0, 4.0, 0.5, 2.0]])
    f = sae.encode(x)
assert f.shape == (1, 10)
assert f[0].tolist() == [5.0, 0, 3.0, 4.0, 0, 0, 0, 0, 0, 0], f"got {f[0].tolist()}"
assert (sae.encode(-x) >= 0).all(), "apply ReLU to the kept values"
print("OK")

`fit_sae` trains an SAE with the reconstruction loss (summed over dimensions), plus the L1 penalty for the ReLU SAE (weighted by decoder norms, which are 1). Decoder rows are renormalized to unit length after every step, so the SAE can't cheat the L1 penalty with small $f$ and large $W_{dec}$. The L1 weight gives about 16 active features per image, the same as $k = 16$.

In [ ]:
SAE_FEATURES = 4096


def fit_sae(sae: nn.Module, l1: float = 0.0, epochs: int = 30) -> nn.Module:
    transformers.set_seed(0)
    sae = sae.to(device)
    opt = torch.optim.Adam(sae.parameters(), lr=4e-4)
    X = Z_food["train"]
    start = time.time()
    for epoch in range(epochs):
        for batch in torch.randperm(len(X), device=device).split(1024):
            x = X[batch]
            reconstruction, f = sae(x)
            loss = ((reconstruction - x) ** 2).sum(1).mean() + l1 * f.sum(1).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            with torch.no_grad():
                sae.W_dec.data = F.normalize(sae.W_dec.data, dim=1)
    sae.eval()
    with torch.no_grad():
        x = Z_food["test"]
        reconstruction, f = sae(x)
        fvu = ((reconstruction - x) ** 2).sum() / ((x - x.mean(dim=0)) ** 2).sum()
        dead = ((f > 0).sum(dim=0) == 0).float().mean()
    print(
        f"{type(sae).__name__}: fraction of variance unexplained {fvu:.3f}, "
        f"active features per image {(f > 0).sum(dim=1).float().mean():.1f}, "
        f"dead features {100 * dead:.1f}%, {time.time() - start:.0f} s"
    )
    return sae


relu_sae = fit_sae(ReLUSAE(384, SAE_FEATURES), l1=4.0)
food_sae = fit_sae(TopKSAE(384, SAE_FEATURES, k=16))

At the same sparsity, TopK reconstructs the embeddings much better: L1 shrinks every active feature, so the ReLU SAE systematically underestimates them. What are the features? For features whose top images come from **different dishes**, i.e. concepts beyond the labels, we show the test images with their largest activations.

In [ ]:
food_test = load_dataset("ethz/food101", split="validation")
with torch.no_grad():
    activations = food_sae.encode(Z_food["test"]).cpu()
top_images = activations.topk(12, dim=0).indices.T  # (features, 12)
labels_of_top = y_food["test"][top_images.numpy()]
num_dishes = np.array([len(set(row)) for row in labels_of_top])
frequency = (activations > 0).float().mean(dim=0).numpy()

# features active on 0.1-2% of images, whose top 12 images come from at least 6 dishes
candidates = np.flatnonzero(
    (num_dishes >= 6) & (frequency > 0.001) & (frequency < 0.02)
)
print(f"{len(candidates)} candidate features out of {SAE_FEATURES}")
shown_features = rng.choice(candidates, 8, replace=False)


def show_feature_images(features: np.ndarray, columns: int = 7) -> None:
    fig, axes = plt.subplots(
        len(features), columns, figsize=(1.9 * columns, 2.1 * len(features))
    )
    for row, feature in zip(axes, features):
        for ax, i in zip(row, top_images[feature][:columns].tolist()):
            ax.imshow(food_test[i]["image"].convert("RGB").resize((160, 160)))
            ax.set_title(food_classes[y_food["test"][i]], fontsize=8)
            ax.axis("off")
        row[0].text(
            -20, 80, f"#{feature}", rotation=90, va="center", ha="right", fontsize=10
        )
    plt.tight_layout()
    plt.show()


show_feature_images(shown_features)

Most are recognizable visual concepts across dishes: a specific ingredient, texture, color, shape or way of serving. Nobody told the model about them: DINOv3 learned them from images alone, and the SAE made them readable. SAEs are used this way to audit vision models, e.g. to find spurious features like watermarks or hospital-specific markers in medical images.

It also works in reverse: what does the model "see" in one image? Below, the features most active on a test image, each with its own top images.

In [ ]:
image_index = 700
features = activations[image_index].topk(4).indices.numpy()
plt.figure(figsize=(3, 3))
plt.imshow(food_test[image_index]["image"].convert("RGB"))
plt.title(
    f"{food_classes[y_food['test'][image_index]]}: features {features.tolist()}",
    fontsize=9,
)
plt.axis("off")
plt.show()
show_feature_images(features)

food_test = relu_sae = None
free_memory()

# Part 6: U-Net

**Dense prediction** tasks, like segmentation, depth estimation or denoising, need an output for every pixel, with both global context (what is this?) and precise localization (where exactly is its boundary?). A downsampling encoder gets the context, but loses spatial detail, which a decoder alone can't recover.

[U-Net](https://arxiv.org/abs/1505.04597) (Ronneberger et al., 2015) is an encoder-decoder with **skip connections**: at every resolution, the encoder features are concatenated with the upsampled decoder features. Only the context passes through the bottleneck; positions and edges go around it. So nothing forces compression, and U-Net isn't an autoencoder. It's the standard architecture for biomedical segmentation and, with noise on the input (a denoising autoencoder again), the core of image restoration and diffusion models.

As in Labs 1 and 2, we build it step by step:

| Step | Change | You implement |
|---|---|---|
| 1 | encoder-decoder without skip connections | Exercise 14: U-Net |
| 2 | skip connections: U-Net | |
| 3 | Dice loss | Exercise 15: Dice loss |
| 4 | data augmentation | |
| 5 | denoising U-Net, residual learning, noise conditioning | Exercise 16: noise-conditioned denoiser |

## 6.1 Retinal vessel segmentation

The retina is the only place where blood vessels can be seen without surgery. Their width, branching and shape change with diabetic retinopathy, hypertension and other diseases, which makes vessel segmentation a classic medical imaging task. Its standard benchmark, [DRIVE](https://drive.grand-challenge.org/) (2004), has color fundus photographs with manually drawn vessels, 20 for training and 20 for testing; we use the 512 x 512 version from [MedSegBench](https://zenodo.org/records/13358372). With so few images, augmentation and the right loss matter a lot, and vessels, often 1-3 pixels wide, are a hard test for the details.

In [ ]:
drive = np.load(download(
    "https://zenodo.org/records/13358372/files/drive_512.npz?download=1",
    "drive_512.npz",
))  # fmt: skip

# the 2 validation images are used for training too
drive_x, drive_y = {}, {}
for split, parts in [("train", ["train", "val"]), ("test", ["test"])]:
    images = np.concatenate([drive[f"{p}_images"] for p in parts])
    drive_x[split] = (
        torch.tensor(images).permute(0, 3, 1, 2).float().div(255).to(device)
    )
    labels = np.concatenate([drive[f"{p}_label"] for p in parts])
    drive_y[split] = torch.tensor(labels > 0).long().to(device)
print(
    f"images: {tuple(drive_x['train'].shape)} train, {tuple(drive_x['test'].shape)} test"
)
print(f"vessel pixels: {100 * drive_y['train'].float().mean():.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
axes[0].imshow(drive_x["train"][0].permute(1, 2, 0).cpu())
axes[1].imshow(drive_y["train"][0].cpu(), cmap="gray")
axes[2].imshow(drive_x["train"][0, :, 200:328, 200:328].permute(1, 2, 0).cpu())
for ax, title in zip(axes, ["fundus image", "vessels (manual)", "zoom: thin vessels"]):
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Training tools

`fit_segmentation` trains on random 256 x 256 crops, 8 per batch, for 2000 steps (about a minute on a GPU, 3 minutes on Colab T4), with AdamW and a one-cycle schedule. `loss_fn(logits, y)` gives the loss, and `augment(x, y)` optionally augments a batch. `segmentation_step` trains a model, evaluates the **Dice score** on full test images, $\frac{2 |P \cap T|}{|P| + |T|}$ for predicted and true vessel pixels (the standard metric of medical segmentation, equal to the F1 score of pixels), and plots all steps so far.

This section is collapsed: you only need to run it.

In [ ]:
SEGMENTATION_STEPS = 2000
segmentation_results = {}
drive_mean = drive_x["train"].mean(dim=(0, 2, 3), keepdim=True)
drive_std = drive_x["train"].std(dim=(0, 2, 3), keepdim=True)


def fit_segmentation(
    model: nn.Module,
    loss_fn: Callable,
    augment: Callable | None = None,
    steps: int = SEGMENTATION_STEPS,
) -> nn.Module:
    transformers.set_seed(0)
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(opt, 2e-3, total_steps=steps)
    generator = np.random.default_rng(0)
    start = time.time()
    for step in range(steps):
        model.train()
        i = torch.randint(len(drive_x["train"]), (8,), device=device)
        r, c = generator.integers(0, 512 - 256 + 1, size=2)
        x = drive_x["train"][i, :, r : r + 256, c : c + 256]
        y = drive_y["train"][i, r : r + 256, c : c + 256]
        if augment is not None:
            x, y = augment(x, y)
        with autocast():
            logits = model((x - drive_mean) / drive_std).float()
        loss = loss_fn(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        scheduler.step()
    print(f"trained in {time.time() - start:.0f} s")
    return model.eval()


@torch.no_grad()
def predict_vessels(model: nn.Module) -> torch.Tensor:
    with autocast():
        x = (drive_x["test"] - drive_mean) / drive_std
        return torch.cat([model(batch).argmax(dim=1) for batch in x.split(4)])


def dice_score(prediction: torch.Tensor, target: torch.Tensor) -> float:
    # mean Dice of the vessel class over test images
    p, t = prediction == 1, target == 1
    dice = 2 * (p & t).sum(dim=(1, 2)) / (p.sum(dim=(1, 2)) + t.sum(dim=(1, 2)))
    return dice.float().mean().item()


def segmentation_step(
    name: str, model: nn.Module, loss_fn: Callable, augment: Callable | None = None
) -> nn.Module:
    print(f"{name}: {count_params(model) / 1e6:.1f}M parameters")
    model = fit_segmentation(model, loss_fn, augment)
    dice = dice_score(predict_vessels(model), drive_y["test"])
    segmentation_results[name] = {"Dice": dice}
    print(f"test Dice: {100 * dice:.1f}%")
    plot_results(segmentation_results, "DRIVE retinal vessels, test set", xlim=(60, 90))
    return model

## Step 1: Encoder-decoder without skip connections

### Exercise 14: U-Net

Implement the U-Net with optional skip connections, using `conv_block` from Part 3. For widths `(32, 64, 128, 256, 512)`:

* encoder: `self.down` has a `conv_block` for each of the first 4 widths, each followed by 2x2 max pooling in `forward` ($512 \to 32$ pixels)
* bottleneck: `self.middle`, a `conv_block` to the last width
* decoder: for each of the first 4 widths in reverse order, `self.up` has a 2x2 transposed convolution with stride 2 from the previous width (`upsample("transposed", ...)` from Part 3), and `self.decode` has a `conv_block`. With skip connections, the upsampled features are **concatenated** with the encoder features of the same resolution, doubling the input channels of the `conv_block`
* `self.head`: a 1x1 convolution to `out_channels`

`forward` saves the output of every encoder block (before pooling) and uses them in reverse order in the decoder.

In [ ]:
class UNet(nn.Module):
    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 2,
        widths: tuple[int, ...] = (32, 64, 128, 256, 512),
        skip_connections: bool = True,
    ) -> None:
        super().__init__()
        self.skip_connections = skip_connections
        ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        ...

In [ ]:
model = UNet().eval()
x = torch.randn(2, 3, 64, 64)
assert model(x).shape == (2, 2, 64, 64)
assert count_params(model) == 7763074, f"expected 7763074, got {count_params(model)}"
no_skip = UNet(skip_connections=False).eval()
assert count_params(no_skip) == 6979714, (
    f"expected 6979714 parameters without skips, got {count_params(no_skip)}"
)

# a change of one input pixel should change the output most at the same position
x2 = x.clone()
x2[:, :, 32, 32] += 10
with torch.no_grad():
    change = (model(x2) - model(x)).abs().sum(dim=(0, 1))
assert change[32, 32] > change[0, 0], "check the skip connections"
print("OK")

In [ ]:
def cross_entropy(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return F.cross_entropy(logits, y)


no_skip_model = segmentation_step(
    "1. no skip connections", UNet(skip_connections=False), cross_entropy
)

## Step 2: Skip connections

In [ ]:
unet = segmentation_step("2. U-Net", UNet(), cross_entropy)

In [ ]:
def show_vessels(
    models: dict,
    crop: tuple[slice, slice] = (slice(150, 350), slice(250, 450)),
    image: int = 1,
) -> None:
    columns = {"image": None, "ground truth": drive_y["test"]} | {
        n: predict_vessels(m) for n, m in models.items()
    }
    fig, axes = plt.subplots(1, len(columns), figsize=(3.6 * len(columns), 3.8))
    for ax, (name, masks) in zip(axes, columns.items()):
        if masks is None:
            ax.imshow(drive_x["test"][image][:, *crop].permute(1, 2, 0).cpu())
        else:
            ax.imshow(masks[image][crop].cpu(), cmap="gray")
            if name != "ground truth":
                dice = dice_score(masks[[image]], drive_y["test"][[image]])
                name += f", Dice {100 * dice:.1f}%"
        ax.set_title(name, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


show_vessels({"no skip connections": no_skip_model, "U-Net": unet})

Without skip connections, 1-pixel wide vessels must be reconstructed from a map downsampled 16 times: they come out blurred and thickened, and the thinnest ones are missing. Skip connections give the decoder the encoder features at full resolution, and the thin vessels appear.

## Step 3: Dice loss

Only 9% of pixels are vessels, and cross-entropy treats every pixel equally, so the model is biased toward the background: uncertain thin vessels are better left out. The **Dice loss** ([Milletari et al., 2016](https://arxiv.org/abs/1606.04797)) optimizes the overlap directly, as a differentiable Dice score with predicted probabilities $p$ and the binary target $t$:

$$\mathcal{L}_{Dice} = 1 - \frac{2 \sum p t + 1}{\sum p + \sum t + 1}$$

with sums over all pixels of the batch, and $+1$ for numerical stability. Being relative to the object size, it's insensitive to class imbalance. It's usually combined with cross-entropy, which gives smoother gradients, as in [nnU-Net](https://www.nature.com/articles/s41592-020-01008-z).

### Exercise 15: Dice loss

Implement `dice_ce_loss(logits, y)`: cross-entropy plus the Dice loss of the vessel class, where $p$ is the softmax probability of class 1, `(B, H, W)`.

In [ ]:
def dice_ce_loss(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    ...

In [ ]:
y = torch.zeros(2, 8, 8, dtype=torch.long)
y[:, 2:4, 2:6] = 1
# confident and correct
perfect = torch.stack([(y == 0).float(), (y == 1).float()], dim=1) * 40 - 20
assert dice_ce_loss(perfect, y) < 0.01, (
    "a perfect confident prediction should have a loss close to 0"
)
uniform = torch.zeros(2, 2, 8, 8)
expected = math.log(2) + 1 - (2 * 0.5 * 16 + 1) / (0.5 * 128 + 16 + 1)
assert abs(dice_ce_loss(uniform, y).item() - expected) < 1e-5, (
    "Dice loss should be computed over the whole batch"
)

unet_dice = segmentation_step("3. + Dice loss", UNet(), dice_ce_loss)
print("OK")

## Step 4: Data augmentation

With 20 training images, the model sees every vessel many times, and overfits. Retinal images can be rotated and flipped freely, and their brightness and contrast vary between cameras. Geometric transformations are applied to images and masks alike, intensity changes to images only.

In [ ]:
def augment_fundus(
    x: torch.Tensor, y: torch.Tensor
) -> tuple[torch.Tensor, torch.Tensor]:
    # the same random rotation by 90 degrees and flip for the whole batch
    k = int(torch.randint(4, (1,)))
    x, y = torch.rot90(x, k, dims=(2, 3)), torch.rot90(y, k, dims=(1, 2))
    if torch.rand(1) < 0.5:
        x, y = x.flip(-1), y.flip(-1)
    # random gamma (contrast) and brightness, different for every image
    gamma = torch.empty(len(x), 1, 1, 1, device=x.device).uniform_(0.8, 1.25)
    brightness = torch.empty(len(x), 1, 1, 1, device=x.device).uniform_(-0.1, 0.1)
    return x.clamp(1e-4, 1) ** gamma + brightness, y


unet_final = segmentation_step(
    "4. + augmentation", UNet(), dice_ce_loss, augment_fundus
)
show_vessels(
    {
        "no skip connections": no_skip_model,
        "U-Net, cross-entropy": unet,
        "U-Net + Dice + augmentation": unet_final,
    }
)

Skip connections and augmentation give the largest gains; the Dice loss adds a small but consistent improvement (about 0.5 points with augmentation, over 2 seeds in our experiments). The final model is close to published results on DRIVE (Dice about 80-83%), after a few minutes of training. Residual blocks (ResUNet), attention gates (Attention U-Net) and a pretrained ResNet-34 encoder didn't improve the result, in line with [nnU-Net](https://www.nature.com/articles/s41592-020-01008-z): a plain U-Net with a well-tuned training recipe (loss, augmentation, patch size, schedule) beats most architectural modifications in medical segmentation.

In [ ]:
no_skip_model = unet = unet_dice = unet_final = None
drive_x = drive_y = None
free_memory()

## 6.2 Denoising U-Net

Back to CT denoising from Part 3. Trained on clean inputs, a U-Net learns nothing useful: the skip connections just copy the input. With noise on the input, it's a **denoising autoencoder**: the bottleneck path provides the knowledge of what clean images look like, and the skip connections the details. We train it like the conv DAE ($\sigma = 0.1$, 2000 steps), with and without skip connections, and add two improvements:

* **residual learning** ([DnCNN](https://arxiv.org/abs/1608.03981)) - the network predicts the noise $n$, and the output is $\tilde{x} - n$. The identity is now the default, and only the correction is learned
* **noise conditioning** - training on random noise levels $\sigma \in [0, 0.2]$, with $\sigma$ as an additional input channel, as in DRUNet. Diffusion models condition their denoising U-Net on the noise level (timestep) the same way, usually with an embedding instead of an input channel

### Exercise 16: Noise-conditioned residual denoiser

Implement `NoiseConditionedDenoiser`, which wraps a U-Net with 2 input channels and 1 output channel. In `forward(x, sigma)`, with `x` of shape `(B, 1, H, W)` and `sigma` of shape `(B,)`:

1. broadcast `sigma` to a noise level map `(B, 1, H, W)`
2. concatenate it with `x` along channels, and predict the noise with the U-Net
3. return `x` minus the predicted noise

`noise_range` (given) samples a different $\sigma$ for every image of a training batch, and passes it to the model.

In [ ]:
DENOISING_WIDTHS = (32, 64, 128, 256)


class NoiseConditionedDenoiser(nn.Module):
    def __init__(self, widths: tuple[int, ...] = DENOISING_WIDTHS) -> None:
        super().__init__()
        self.unet = UNet(2, 1, widths)

    def forward(self, x: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
        ...


class ResidualDenoiser(nn.Module):
    # predicts the noise, for a fixed noise level
    def __init__(self, widths: tuple[int, ...] = DENOISING_WIDTHS) -> None:
        super().__init__()
        self.unet = UNet(1, 1, widths)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x - self.unet(x)


def noise_range(x: torch.Tensor, max_sigma: float = 0.2) -> tuple[torch.Tensor, dict]:
    sigma = torch.rand(len(x), device=x.device) * max_sigma
    return x + sigma.view(-1, 1, 1, 1) * torch.randn_like(x), {"sigma": sigma}

In [ ]:
model = NoiseConditionedDenoiser().eval()
x = torch.rand(3, 1, 32, 32)
sigma = torch.tensor([0.0, 0.1, 0.2])
assert model(x, sigma).shape == x.shape
with torch.no_grad():
    nn.init.zeros_(model.unet.head.weight)
    nn.init.zeros_(model.unet.head.bias)
assert torch.allclose(model(x, sigma), x), (
    "with zero predicted noise, the output should equal the input"
)
with torch.no_grad():
    model.unet.head.bias.fill_(0.5)
assert torch.allclose(model(x, sigma), x - 0.5), "subtract the predicted noise"
# the noise level must change the output
model = NoiseConditionedDenoiser().eval()
assert not torch.allclose(model(x, torch.zeros(3)), model(x, torch.full((3,), 0.2))), (
    "use sigma as input"
)
print("OK")

Training the 4 U-Nets takes about 4 minutes on a GPU (12 minutes on Colab T4). We evaluate them, and the conv DAE from Part 3, at noise levels from 0.02 to 0.2.

In [ ]:
NOISE_LEVELS = [0.02, 0.05, 0.1, 0.15, 0.2]
unit_noise = fixed_noise(ct_clean.shape, seed=1)
noisy_levels = [ct_clean + s * unit_noise for s in NOISE_LEVELS]


def plot_psnr(curves: dict, title: str) -> None:
    # PSNR at every noise level; the dotted line is the training noise level
    fig, ax = plt.subplots(figsize=(8, 4))
    for name, values in curves.items():
        ls = "--" if name == "noisy input" else "-"
        ax.plot(NOISE_LEVELS, values, marker="o", ls=ls, lw=2, label=name)
    ax.axvline(CT_SIGMA, color="gray", ls=":")
    ax.set(xlabel="noise level sigma", ylabel="PSNR (dB)", title=title)
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    plt.show()


denoisers = {
    "conv DAE (Part 3)": conv_dae,
    "no skip connections": fit_denoiser(
        UNet(1, 1, DENOISING_WIDTHS, skip_connections=False)
    ),
    "U-Net": fit_denoiser(UNet(1, 1, DENOISING_WIDTHS)),
    "U-Net, residual": fit_denoiser(ResidualDenoiser()),
    "U-Net, residual, noise-conditioned": fit_denoiser(
        NoiseConditionedDenoiser(), noise=noise_range
    ),
}
denoising_psnr = {"noisy input": [psnr(x, ct_clean) for x in noisy_levels]}
for name, model in denoisers.items():
    conditioned = isinstance(model, NoiseConditionedDenoiser)
    denoising_psnr[name] = [
        psnr(denoise(model, x, sigma=s if conditioned else None), ct_clean)
        for s, x in zip(NOISE_LEVELS, noisy_levels)
    ]

table = pd.DataFrame(denoising_psnr, index=[f"sigma={s}" for s in NOISE_LEVELS])
print(table.T.round(2).to_string())
plot_psnr(denoising_psnr, "CT denoising (training noise level: dotted line)")

In [ ]:
show_ct(
    {
        "clean": ct_clean,
        "noisy": ct_noisy,
        "conv DAE (Part 3)": denoise(conv_dae, ct_noisy),
        "U-Net, residual, conditioned": denoise(
            denoisers["U-Net, residual, noise-conditioned"], ct_noisy, sigma=CT_SIGMA
        ),
        "DRUNet (pretrained)": ct_drunet,
    },
    [3, 11],
)

Models trained at one noise level always remove that amount of noise: they leave part of stronger noise, and remove details at weaker noise; at very weak noise, the output of the conv DAE is worse than its input. Skip connections help most at low noise, where there are details worth keeping: the U-Net is 3 dB better than the deep network without them. At the training noise level, the shallow conv DAE is as good as the U-Net, since these CT slices are fairly smooth, and residual learning improves the U-Net further. One noise-conditioned model is the best or close to the best across the whole range, losing half a dB at the training noise level, since it has to handle all of them. With under 2M parameters and a minute of training, it gets within about 1 dB of DRUNet, a 33M model trained on a large dataset of natural images.

This is the building block of **diffusion models** ([Ho et al., 2020](https://arxiv.org/abs/2006.11239)): a noise-conditioned denoising U-Net, applied repeatedly from pure noise to an image. At high noise levels the bottleneck path matters most (global structure), at low noise levels the skip connections (details).

# Summary

All models in this lab are encoder -> bottleneck -> decoder; they differ in how the identity mapping is made impossible, and what the code is used for:

| Type | Bottleneck / regularizer | Used for (in this lab) |
|---|---|---|
| PCA, linear AE | narrow linear layer | compression baseline |
| MLP autoencoder | narrow nonlinear code, task-specific likelihood | representation learning, visualization, novelty detection |
| denoising AE | noise on the input | restoration, regularization |
| masked AE | 75% of the input removed | self-supervised pretraining |
| sparse AE | wide, sparse code | interpretability, steering |
| U-Net | none (skip connections) | dense prediction; with noise a denoising AE |

Not covered here: **variational autoencoders** (VAE, a probabilistic latent; the image compressor of Stable Diffusion), **vector-quantized** autoencoders (VQ-VAE, discrete codes; the basis of image and audio tokenizers), and diffusion models themselves.